In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# =========================================================
# Paths and Settings
# =========================================================
from pathlib import Path
from collections import Counter, defaultdict
import ast
import json
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories/facial_skincare")

ITEMS_PATH = PROJECT_ROOT / "data" / "raw" / "items_Skin_Care_Face_W2_2019_2023.parquet"
REVIEWS_PATH = PROJECT_ROOT / "data" / "raw" / "reviews_Skin_Care_Face_W2_2019_2023.parquet"

PROCESSED_ITEMS_DIR = PROJECT_ROOT / "data" / "processed" / "items"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "stage0_feature_engineering"

SCHEMA_OUTPUT_PATH = PROCESSED_ITEMS_DIR / "face_item_schema.parquet"
SCHEMA_NORM_OUTPUT_PATH = PROCESSED_ITEMS_DIR / "face_item_schema_norm.parquet"
SCHEMA_FULL_OUTPUT_PATH = PROCESSED_ITEMS_DIR / "face_item_schema_full.parquet"
COMMON_ITEM_SCHEMA_PATH = PROCESSED_ITEMS_DIR / "face_item_schema_common.parquet"
ITEM_FACETS_PATH = PROCESSED_ITEMS_DIR / "face_items_facets.parquet"
FACET_VOCAB_PATH = PROCESSED_ITEMS_DIR / "face_facet_vocab.parquet"

EVALUATION_WINDOW_MONTHS = 9
EVALUATION_WINDOW_END = pd.Timestamp("2023-09-12T14:52:26.427000+00:00")
EVALUATION_WINDOW_START = (
    EVALUATION_WINDOW_END - pd.DateOffset(months=EVALUATION_WINDOW_MONTHS)
)
TRAIN_REVIEW_CUTOFF_EXCLUSIVE = EVALUATION_WINDOW_START
REVIEW_REPUTATION_SOURCE = "historical_reviews_before_train_cutoff"

PROCESSED_ITEMS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", ITEMS_PATH)
print("Input:", REVIEWS_PATH)
print("Output:", SCHEMA_OUTPUT_PATH)


In [ ]:
# =========================================================
# Item Metadata Loading
# =========================================================
items = pd.read_parquet(ITEMS_PATH)

items = items.drop(columns=["bought_together"], errors="ignore")
items = items.drop(columns=["average_rating", "rating_number", "price"], errors="ignore")

items["parent_asin"] = items["parent_asin"].astype(str).str.strip()
items = items[items["parent_asin"].str.len() > 0].copy()

RAW_GENERIC_SUB_CATEGORY_LABELS = {"Beauty & Personal Care", "Skin Care", "Face"}

def _raw_category_text(x):
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None
    text = str(x).strip()
    return text if text else None

def _raw_category_token(x):
    text = _raw_category_text(x)
    if text is None:
        return None
    text = text.lower().replace("&", " and ").replace("_", " ").replace("-", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else None

RAW_GENERIC_SUB_CATEGORY_LABEL_TOKENS = {
    _raw_category_token(label)
    for label in RAW_GENERIC_SUB_CATEGORY_LABELS
}

RAW_CATEGORY_PATH_PATTERN = re.compile(r"\s*(?:\|\|\||\||>|;)\s*|\s+/\s+")

def _parse_raw_category_values(x):
    if isinstance(x, (list, tuple, set)):
        return list(x)

    if isinstance(x, str):
        stripped = x.strip()
        if not stripped:
            return []

        if stripped.startswith("[") and stripped.endswith("]"):
            try:
                parsed = ast.literal_eval(stripped)
            except (ValueError, SyntaxError, TypeError, json.JSONDecodeError):
                parsed = json.loads(stripped)

            if isinstance(parsed, (list, tuple, set)):
                return list(parsed)

        return [stripped]

    text = _raw_category_text(x)
    return [text] if text else []

def _extract_category_leaf_values(x):
    leaves = []
    seen = set()

    for part in _parse_raw_category_values(x):
        part_text = _raw_category_text(part)
        if part_text is None:
            continue

        tokens = [
            _raw_category_text(token)
            for token in RAW_CATEGORY_PATH_PATTERN.split(part_text)
        ]
        tokens = [token for token in tokens if token]

        specific_tokens = [
            token
            for token in tokens
            if _raw_category_token(token) not in RAW_GENERIC_SUB_CATEGORY_LABEL_TOKENS
        ]

        if not specific_tokens:
            continue

        leaf = specific_tokens[-1]
        key = leaf.lower()

        if key in seen:
            continue

        seen.add(key)
        leaves.append(leaf)

    return leaves

items["categories_leaf"] = items["categories"].apply(_extract_category_leaf_values)
items["categories_leaf_text"] = items["categories_leaf"].apply(
    lambda values: " | ".join(values) if values else None
)

pass
pass
pass

In [ ]:
# =========================================================
# Normalization Helpers
# =========================================================
def clean_text(x):
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    text = str(x).strip().lower()
    text = text.replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_raw_text(x):
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None
    x = str(x).strip()
    x = x.replace("\n", " ").replace("\t", " ")
    x = re.sub(r"\s+", " ", x)
    return x if x else None

def parse_maybe_dict(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return None
        try:
            parsed = ast.literal_eval(x)
            return parsed if isinstance(parsed, dict) else None
        except (ValueError, SyntaxError, TypeError, json.JSONDecodeError):
            try:
                parsed = json.loads(x)
                return parsed if isinstance(parsed, dict) else None
            except (ValueError, SyntaxError, TypeError, json.JSONDecodeError):
                return None
    return None

def parse_maybe_list(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, (list, tuple, set)):
        return list(x)
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return None
        try:
            parsed = ast.literal_eval(x)
            return list(parsed) if isinstance(parsed, (list, tuple, set)) else None
        except (ValueError, SyntaxError, TypeError, json.JSONDecodeError):
            try:
                parsed = json.loads(x)
                return list(parsed) if isinstance(parsed, list) else None
            except (ValueError, SyntaxError, TypeError, json.JSONDecodeError):
                return None
    return None

def normalize_key(k):
    if k is None:
        return None
    k = str(k).strip()
    k = k.replace("\n", " ").replace("\t", " ")
    k = " ".join(k.split())
    return k

def normalize_value(v):
    if v is None:
        return None
    if isinstance(v, (dict, list, tuple, set)):
        return json.dumps(v, ensure_ascii=False, sort_keys=True)
    v = str(v).strip()
    v = " ".join(v.split())
    return v if v else None

def ensure_text_field(x):
    parsed_dict = parse_maybe_dict(x)
    if isinstance(parsed_dict, dict):
        parts = []
        for k, v in parsed_dict.items():
            k_norm = normalize_raw_text(k)
            v_norm = normalize_raw_text(v)
            if k_norm and v_norm:
                parts.append(f"{k_norm}: {v_norm}")
            elif v_norm:
                parts.append(v_norm)
        return " | ".join(parts) if parts else None

    parsed_list = parse_maybe_list(x)
    if isinstance(parsed_list, list):
        parts = [normalize_raw_text(v) for v in parsed_list]
        parts = [v for v in parts if v]
        return " | ".join(parts) if parts else None

    return normalize_raw_text(x)

pass

In [ ]:
# =========================================================
# Detail Key Mapping
# =========================================================
KEY_MAP = {
    "Brand": "brand",
    "Brand Name": "brand",

    "Item Form": "form",
    "Form Factor": "form",
    "Dosage Form": "form",

    "Skin Type": "skin_type",
    "Skin type": "skin_type",

    "Active Ingredients": "ingredient",
    "Special Ingredients": "ingredient",

    "Product Benefits": "benefit",

    "Scent": "scent",
    "Perfume": "scent",

    "Material Type Free": "free_claim",
    "Material free": "free_claim",
    "Material Feature": "free_claim",

    "Sub Category": "sub_category",
    "Sub-Category": "sub_category",
    "SubCategory": "sub_category",

    "Is Discontinued By Manufacturer": "is_discontinued_by_manufacturer_raw",

    "Use for": "use_for",
    "Recommended Uses For Product": "use_for",
    "Specific Uses For Product": "use_for",

    "Sun Protection": "sun_protection",
    "Sun Protection Factor": "sun_protection",
    "Coverage": "coverage",
}
KEY_MAP = {clean_text(k): v for k, v in KEY_MAP.items()}

In [ ]:
# =========================================================
# Value Category Helpers
# =========================================================
SPLIT_PATTERN = re.compile(r"\s*[,;/|]\s*")
CATEGORY_PATH_PATTERN = re.compile(r"\s*(?:\|\|\||\||>|;)\s*|\s+/\s+")
NULL_LIKE_STRINGS = {"", "none", "null", "nan", "n/a", "na", "[]", "{}"}
GENERIC_SUB_CATEGORY_LABELS = {"Beauty & Personal Care", "Skin Care", "Face"}
GENERIC_SUB_CATEGORY_LABEL_TOKENS = {label.lower() for label in GENERIC_SUB_CATEGORY_LABELS}
SUB_CATEGORY_ALIAS_MAP = {}
DISCONTINUED_TRUE_VALUES = {"yes", "true"}
DISCONTINUED_FALSE_VALUES = {"no", "false"}

def apply_sub_category_alias(x):
    label = clean_text(x)
    if label is None:
        return None
    token = normalize_sub_category_token(label)
    mapped = SUB_CATEGORY_ALIAS_MAP.get(token)
    return mapped if mapped else label

def split_multi_value(v):
    text = clean_text(v)
    if text is None or text.lower() in NULL_LIKE_STRINGS:
        return []
    if len(text) > 200 and not re.search(r"[,;/|]", text):
        return [text]
    parts = [clean_text(part) for part in SPLIT_PATTERN.split(text)]
    parts = [part for part in parts if part and part.lower() not in NULL_LIKE_STRINGS]
    return parts if parts else [text]

def dedupe_keep_order(values):
    seen = set()
    out = []
    for value in values:
        cleaned = clean_text(value)
        if cleaned is None:
            continue
        key = cleaned.lower()
        if key in NULL_LIKE_STRINGS or key in seen:
            continue
        seen.add(key)
        out.append(cleaned)
    return out

def normalize_sub_category_token(x):
    text = clean_text(x)
    if text is None:
        return None
    text = text.lower().replace("&", " and ").replace("_", " ").replace("-", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else None

GENERIC_SUB_CATEGORY_LABEL_TOKENS = {
    normalize_sub_category_token(label)
    for label in GENERIC_SUB_CATEGORY_LABELS
    if normalize_sub_category_token(label)
}

def parse_raw_category_parts(x):
    if isinstance(x, (list, tuple, set)):
        return list(x)
    if isinstance(x, str):
        stripped = x.strip()
        if not stripped:
            return []
        if stripped.startswith("[") and stripped.endswith("]"):
            try:
                parsed = ast.literal_eval(stripped)
            except (ValueError, SyntaxError, TypeError, json.JSONDecodeError):
                try:
                    parsed = json.loads(stripped)
                except (ValueError, SyntaxError, TypeError, json.JSONDecodeError):
                    parsed = None
            if isinstance(parsed, (list, tuple, set)):
                return list(parsed)
        return [stripped]
    text = clean_text(x)
    return [text] if text else []

def extract_category_leaf_values(x):
    leaves = []
    for part in parse_raw_category_parts(x):
        part_text = clean_text(part)
        if part_text is None:
            continue
        tokens = [clean_text(token) for token in CATEGORY_PATH_PATTERN.split(part_text)]
        tokens = [token for token in tokens if token and token.lower() not in NULL_LIKE_STRINGS]
        if not tokens:
            continue
        specific_tokens = [
            token
            for token in tokens
            if normalize_sub_category_token(token) not in GENERIC_SUB_CATEGORY_LABEL_TOKENS
        ]
        leaf = specific_tokens[-1] if specific_tokens else None
        if leaf:
            leaves.append(leaf)
    return dedupe_keep_order(leaves)

def sanitize_sub_category_values(values):
    if isinstance(values, (list, tuple, set)):
        raw_values = list(values)
    else:
        raw_values = [values]
    sanitized = []
    for value in raw_values:
        sanitized.extend(extract_category_leaf_values(value))
    return dedupe_keep_order(sanitized)

def clean_sub_category_label(x):
    leaf_values = sanitize_sub_category_values(x)
    label = clean_text(leaf_values[-1] if leaf_values else x)
    if label is None:
        return None
    label = label.replace("_", " ")
    label = re.sub(r"\s+", " ", label).strip(" -")
    token = normalize_sub_category_token(label)
    if token is None or token in NULL_LIKE_STRINGS or token in GENERIC_SUB_CATEGORY_LABEL_TOKENS:
        return None
    return apply_sub_category_alias(label)

def normalize_discontinued_value(x):
    text = clean_text(x)
    if text is None:
        return None, False
    key = text.strip().lower()
    if key in DISCONTINUED_TRUE_VALUES:
        return text, True
    if key in DISCONTINUED_FALSE_VALUES:
        return text, False
    return text, False

def parse_raw_categories(x):
    return extract_category_leaf_values(x)

def extract_sub_category_from_categories(x):
    raw_values = parse_raw_categories(x)
    cleaned_values = [clean_sub_category_label(value) for value in raw_values]
    cleaned_values = [value for value in cleaned_values if value]
    return dedupe_keep_order(cleaned_values)

def merge_sub_category_sources(details_values, categories_values):
    merged = []
    for values in (details_values, categories_values):
        for value in sanitize_sub_category_values(values):
            cleaned = clean_sub_category_label(value)
            if cleaned:
                merged.append(cleaned)
    return dedupe_keep_order(merged)

In [ ]:
# =========================================================
# Metadata Extraction
# =========================================================
DROP_RAW_KEYS = set()

def normalize_details_row(details_obj):
    result = {
        "brand": [], "form": [], "skin_type": [], "ingredient": [], "benefit": [],
        "scent": [], "free_claim": [], "sub_category": [], "use_for": [],
        "sun_protection": [], "coverage": [],
        "is_discontinued_by_manufacturer_raw": None,
        "is_discontinued": False,
    }
    dropped = []
    unmapped = []
    if not isinstance(details_obj, dict):
        return result, dropped, unmapped
    for raw_k, raw_v in details_obj.items():
        raw_k_clean = clean_text(raw_k)
        if raw_k_clean is None:
            continue
        if raw_k_clean in DROP_RAW_KEYS:
            raw_v_clean = clean_text(raw_v)
            if raw_v_clean is not None:
                dropped.append({"raw_key": raw_k_clean, "raw_value": raw_v_clean})
            continue
        mapped_key = KEY_MAP.get(raw_k_clean)
        if mapped_key is None:
            raw_v_clean = clean_text(raw_v)
            if raw_v_clean is not None:
                unmapped.append({"raw_key": raw_k_clean, "raw_value": raw_v_clean})
            continue
        if mapped_key == "sub_category":
            values = [clean_sub_category_label(value) for value in split_multi_value(raw_v)]
            values = [value for value in values if value]
            values = dedupe_keep_order(values)
            if values:
                result[mapped_key].extend(values)
            continue
        if mapped_key == "is_discontinued_by_manufacturer_raw":
            raw_text, is_discontinued = normalize_discontinued_value(raw_v)
            result["is_discontinued_by_manufacturer_raw"] = raw_text
            result["is_discontinued"] = is_discontinued
            continue
        values = dedupe_keep_order(split_multi_value(raw_v))
        if mapped_key in result and values:
            result[mapped_key].extend(values)
    for key in ["brand", "form", "skin_type", "ingredient", "benefit", "scent", "free_claim", "sub_category", "use_for", "sun_protection", "coverage"]:
        result[key] = dedupe_keep_order(result[key])
    return result, dropped, unmapped

In [ ]:
# =========================================================
# Metadata Normalization
# =========================================================
normalized_rows = []
dropped_rows = []
unmapped_rows = []
for idx, x in items["details"].items():
    d = parse_maybe_dict(x)
    norm, dropped, unmapped = normalize_details_row(d)
    row = {"row_index": idx}
    for k, vals in norm.items():
        if k == "is_discontinued_by_manufacturer_raw":
            row[k] = vals
        elif k == "is_discontinued":
            row[k] = bool(vals)
        else:
            row[k] = vals
            row[f"{k}_text"] = " | ".join(vals) if vals else None
    normalized_rows.append(row)
    for obj in dropped:
        dropped_rows.append({"row_index": idx, "raw_key": obj["raw_key"], "raw_value": obj["raw_value"]})
    for obj in unmapped:
        unmapped_rows.append({"row_index": idx, "raw_key": obj["raw_key"], "raw_value": obj["raw_value"]})
details_norm_df = pd.DataFrame(normalized_rows)
details_dropped_df = pd.DataFrame(dropped_rows)
details_unmapped_df = pd.DataFrame(unmapped_rows)
if "is_discontinued_by_manufacturer_raw" in details_norm_df.columns:
    details_norm_df["is_discontinued_by_manufacturer_raw"] = details_norm_df["is_discontinued_by_manufacturer_raw"].apply(clean_text)
if "is_discontinued" in details_norm_df.columns:
    details_norm_df["is_discontinued"] = details_norm_df["is_discontinued"].fillna(False).astype(bool)
discontinued_value_counts_details_df = pd.DataFrame([
    {"is_discontinued_by_manufacturer_raw": raw_value, "count": int(count)}
    for raw_value, count in details_norm_df["is_discontinued_by_manufacturer_raw"].fillna("<missing>").value_counts(dropna=False).items()
]) if "is_discontinued_by_manufacturer_raw" in details_norm_df.columns else pd.DataFrame(columns=["is_discontinued_by_manufacturer_raw", "count"])
pass
pass
pass
pass
pass
pass
pass
pass

In [ ]:
# =========================================================
# Metadata Schema Base
# =========================================================
items_details = items.copy()
items_details = items_details.merge(details_norm_df, left_index=True, right_on="row_index", how="left")
items_details = items_details.drop(columns=["row_index"])
if "sub_category" not in items_details.columns:
    items_details["sub_category"] = [[] for _ in range(len(items_details))]
else:
    items_details["sub_category"] = items_details["sub_category"].apply(lambda x: x if isinstance(x, list) else [])
if "is_discontinued" in items_details.columns:
    items_details["is_discontinued"] = items_details["is_discontinued"].fillna(False).astype(bool)
if "categories_leaf" in items_details.columns:
    category_sub_category_values = items_details["categories_leaf"].apply(extract_sub_category_from_categories)
elif "categories" in items_details.columns:
    category_sub_category_values = items_details["categories"].apply(extract_sub_category_from_categories)
else:
    category_sub_category_values = pd.Series([[] for _ in range(len(items_details))], index=items_details.index)
items_details["sub_category_details"] = items_details["sub_category"].apply(lambda x: sanitize_sub_category_values(x))
items_details["sub_category_raw"] = [merge_sub_category_sources(a, b) for a, b in zip(items_details["sub_category_details"], category_sub_category_values)]
items_details["sub_category"] = items_details["sub_category_raw"]
items_details["sub_category_text"] = items_details["sub_category_raw"].apply(lambda values: " | ".join(values) if isinstance(values, list) and values else None)
for col in ["features", "description", "details", "categories"]:
    if col in items_details.columns:
        items_details[col] = items_details[col].apply(ensure_text_field)
pass
pass
pass
pass

In [ ]:
# =========================================================
# Brand Metadata
# =========================================================
def first_non_empty(a, b):
    a = None if pd.isna(a) else str(a).strip()
    b = None if pd.isna(b) else str(b).strip()
    if a:
        return a
    if b:
        return b
    return None

items_details["brand_meta"] = [
    first_non_empty(a, b)
    for a, b in zip(items_details.get("brand_x"), items_details.get("brand_text"))
]

pass
pass
pass

pass

In [ ]:
# =========================================================
# Coverage Audit
# =========================================================
summary_rows = []
target_fields = ["form", "sub_category", "sub_category_raw", "ingredient", "benefit", "skin_type", "scent", "free_claim", "use_for", "sun_protection", "coverage"]
available_fields = [c for c in target_fields if c in items_details.columns]
pass
for col in available_fields:
    non_empty = items_details[col].apply(lambda x: isinstance(x, list) and len(x) > 0).sum()
    summary_rows.append({"field": col, "non_empty_rows": int(non_empty), "coverage_ratio": round(non_empty / len(items_details), 6)})
if "brand_meta" in items_details.columns:
    summary_rows.append({"field": "brand_meta", "non_empty_rows": int(items_details["brand_meta"].notna().sum()), "coverage_ratio": round(items_details["brand_meta"].notna().mean(), 6)})
if "sub_category_text" in items_details.columns:
    summary_rows.append({"field": "sub_category_text", "non_empty_rows": int(items_details["sub_category_text"].notna().sum()), "coverage_ratio": round(items_details["sub_category_text"].notna().mean(), 6)})
if "is_discontinued_by_manufacturer_raw" in items_details.columns:
    summary_rows.append({"field": "is_discontinued_by_manufacturer_raw", "non_empty_rows": int(items_details["is_discontinued_by_manufacturer_raw"].notna().sum()), "coverage_ratio": round(items_details["is_discontinued_by_manufacturer_raw"].notna().mean(), 6)})
if "is_discontinued" in items_details.columns:
    summary_rows.append({"field": "is_discontinued_true", "non_empty_rows": int(items_details["is_discontinued"].sum()), "coverage_ratio": round(items_details["is_discontinued"].mean(), 6)})
details_field_coverage_df = pd.DataFrame(summary_rows).sort_values(["coverage_ratio", "field"], ascending=[False, True])
pass

In [ ]:
# =========================================================
# Top Value Audit
# =========================================================
from collections import Counter

top_value_rows = []
target_fields = ["form", "sub_category", "sub_category_raw", "ingredient", "benefit", "skin_type", "scent", "free_claim", "use_for", "sun_protection", "coverage"]
available_fields = [c for c in target_fields if c in items_details.columns]
for col in available_fields:
    counter = Counter()
    for vals in items_details[col]:
        if isinstance(vals, list):
            counter.update(vals)
    for rank, (val, cnt) in enumerate(counter.most_common(50), start=1):
        top_value_rows.append({"field": col, "rank": rank, "value": val, "count": cnt})
if "brand_meta" in items_details.columns:
    brand_counter = Counter(items_details["brand_meta"].dropna().astype(str))
    for rank, (val, cnt) in enumerate(brand_counter.most_common(50), start=1):
        top_value_rows.append({"field": "brand_meta", "rank": rank, "value": val, "count": cnt})
details_top_values_df = pd.DataFrame(top_value_rows)
pass
pass

In [ ]:
# =========================================================
# Concern Lexicon
# =========================================================
CONCERN_PATTERNS = {
    "dryness": [
        r"\bdry\b", r"\bdrying\b", r"\bdryness\b", r"\bdehydrated\b", r"\bdehydration\b"
    ],
    "acne": [
        r"\bacne\b", r"\bpimple[s]?\b", r"\bpimples\b", r"\bbreakout[s]?\b", r"\bblemish(?:es)?\b", r"\bclogged pores?\b"
    ],
    "redness": [
        r"\bredness\b", r"\bred\b", r"\bflushing\b"
    ],
    "sensitivity": [
        r"\bsensitive\b", r"\bsensitivity\b", r"\birritat(?:e|ed|ing|ion)\b", r"\bsting(?:ing)?\b", r"\bburn(?:ing|ed)?\b"
    ],
    "pores": [
        r"\bpore\b", r"\bpores\b", r"\bpore size\b", r"\benlarged pores?\b"
    ],
    "oiliness": [
        r"\boily\b", r"\boiliness\b", r"\bgreasy\b", r"\bsebum\b", r"\bshine\b"
    ],
    "wrinkles": [
        r"\bwrinkle[s]?\b", r"\bfine lines?\b", r"\bcrow'?s feet\b", r"\bage lines?\b"
    ],
    "dark_spots": [
        r"\bdark spots?\b", r"\bhyperpigmentation\b", r"\bpigmentation\b", r"\bdiscoloration\b", r"\bpost-acne marks?\b", r"\bacne scars?\b"
    ],
    "dullness": [
        r"\bdull\b", r"\bdullness\b", r"\btired skin\b", r"\blackluster\b"
    ],
    "rosacea": [
        r"\brosacea\b"
    ],
    "eczema": [
        r"\beczema\b"
    ],
    "itching": [
        r"\bitch(?:y|ing)?\b"
    ],
    "swelling": [
        r"\bswelling\b", r"\bswollen\b", r"\bpuff(?:y|iness)\b"
    ],
}

In [ ]:
# =========================================================
# Concern Matching
# =========================================================
compiled_concern_patterns = {
    k: [re.compile(p) for p in patterns]
    for k, patterns in CONCERN_PATTERNS.items()
}


def match_any(patterns, text):
    return any(p.search(text) for p in patterns)


def extract_concerns_from_text(text):
    found = []
    for concern, patterns in compiled_concern_patterns.items():
        if match_any(patterns, text):
            found.append(concern)
    return found

# Metadata-Only Schema Base


In [ ]:
# =========================================================
# Metadata Backbone
# =========================================================
pass
pass

DETAIL_BACKBONE_COLS = [
    "parent_asin", "title", "main_category", "categories", "categories_leaf", "categories_leaf_text",
    "brand_meta", "form", "form_text", "sub_category", "sub_category_text", "sub_category_raw",
    "is_discontinued_by_manufacturer_raw", "is_discontinued",
    "skin_type", "skin_type_text", "ingredient", "ingredient_text", "benefit", "benefit_text",
    "scent", "scent_text", "free_claim", "free_claim_text", "use_for", "use_for_text",
    "details", "features", "description",
]

available_detail_cols = [c for c in DETAIL_BACKBONE_COLS if c in items_details.columns]
items_backbone_df = items_details.loc[:, available_detail_cols].copy()
items_backbone_df["parent_asin"] = items_backbone_df["parent_asin"].astype(str).str.strip()
items_backbone_df = items_backbone_df.loc[items_backbone_df["parent_asin"].str.len().gt(0)]
items_backbone_df = items_backbone_df.drop_duplicates("parent_asin")

pass
pass
pass

In [ ]:
# =========================================================
# Item Identity Check
# =========================================================
item_dup_count = items_backbone_df["parent_asin"].duplicated().sum()

pass

if item_dup_count > 0:
    pass

In [ ]:
# =========================================================
# Concern Claims
# =========================================================
def metadata_value_to_text(value):
    if value is None:
        return ""
    if isinstance(value, float) and pd.isna(value):
        return ""
    if isinstance(value, list):
        return " ".join(metadata_value_to_text(v) for v in value)
    if isinstance(value, dict):
        return " ".join(f"{k} {metadata_value_to_text(v)}" for k, v in value.items())
    return str(value)

metadata_concern_source_cols = [
    "title",
    "categories",
    "categories_leaf",
    "categories_leaf_text",
    "sub_category",
    "sub_category_text",
    "sub_category_raw",
    "form",
    "form_text",
    "skin_type",
    "skin_type_text",
    "ingredient",
    "ingredient_text",
    "benefit",
    "benefit_text",
    "use_for",
    "use_for_text",
    "free_claim",
    "free_claim_text",
    "features",
    "description",
    "details",
]
metadata_concern_source_cols = [c for c in metadata_concern_source_cols if c in items_backbone_df.columns]

def build_metadata_concern_text(row):
    parts = [metadata_value_to_text(row.get(col)) for col in metadata_concern_source_cols]
    return clean_text(" ".join(part for part in parts if part))

def join_concern_claims(values):
    values = [v for v in values if v]
    return " | ".join(values) if values else None

item_schema_base = items_backbone_df.copy()
item_schema_base["concern_claim_norm"] = item_schema_base.apply(
    lambda row: extract_concerns_from_text(build_metadata_concern_text(row)),
    axis=1,
)
item_schema_base["concern_claim_norm_text"] = item_schema_base["concern_claim_norm"].apply(join_concern_claims)

pass
pass
pass
preview_cols = ["parent_asin", "title", "categories", "brand_meta", "form_text", "sub_category_text", "is_discontinued_by_manufacturer_raw", "is_discontinued", "skin_type_text", "ingredient_text", "benefit_text", "scent_text", "free_claim_text", "concern_claim_norm_text"]
preview_cols = [c for c in preview_cols if c in item_schema_base.columns]
pass

In [ ]:
# =========================================================
# Coverage Audit
# =========================================================
coverage_rows = []
target_cols = ["brand_meta", "form_text", "sub_category_text", "is_discontinued_by_manufacturer_raw", "skin_type_text", "ingredient_text", "benefit_text", "scent_text", "free_claim_text", "use_for_text", "concern_claim_norm_text"]
for col in target_cols:
    if col in item_schema_base.columns:
        non_null = item_schema_base[col].notna().sum()
        coverage_rows.append({"column": col, "non_null_rows": int(non_null), "coverage_ratio": round(non_null / len(item_schema_base), 6)})
if "is_discontinued" in item_schema_base.columns:
    coverage_rows.append({"column": "is_discontinued_true", "non_null_rows": int(item_schema_base["is_discontinued"].sum()), "coverage_ratio": round(item_schema_base["is_discontinued"].mean(), 6)})
item_schema_coverage_df = pd.DataFrame(coverage_rows).sort_values("coverage_ratio", ascending=False)
pass

# Final Metadata Schema


In [ ]:
# =========================================================
# Item Schema Assembly
# =========================================================
if "item_schema_base" not in globals():
    raise RuntimeError("Run the source item schema aggregation section first.")

pass
pass
pass

In [ ]:
# =========================================================
# Normalization Dictionaries
# =========================================================

FORM_MAP = {
    "cream": "cream",
    "creme": "cream",
    "face cream": "cream",
    "facial cream": "cream",
    "gel": "gel",
    "gel cream": "gel",
    "oil": "oil",
    "face oil": "oil",
    "facial oil": "oil",
    "serum": "serum",
    "face serum": "serum",
    "facial serum": "serum",
    "liquid": "liquid",
    "lotion": "lotion",
    "face lotion": "lotion",
    "foam": "foam",
    "foam cleanser": "foam",
    "cleansing foam": "foam",
    "mask": "mask",
    "sheet": "sheet",
    "sheet mask": "sheet",
    "sheet masks": "sheet",
    "sheet_mask": "sheet",
    "spray": "spray",
    "mist": "spray",
    "clay": "clay",
    "bar": "bar",
    "powder": "powder",
    "patch": "patch",
    "pads": "pad",
    "pad": "pad",
    "drops": "drop",
    "drop": "drop",
    "essence": "essence",
    "balm": "balm",
    "paste": "paste",
    "stick": "stick",
    "toner": "liquid",
}

SUB_CATEGORY_CANONICAL_MAP = {
    "cleanser": "cleanser",
    "cleansers": "cleanser",
    "face cleanser": "cleanser",
    "facial cleanser": "cleanser",
    "face cleansers": "cleanser",
    "facial cleansers": "cleanser",
    "face wash": "face_wash",
    "washes": "washes",
    "wash": "washes",
    "cleansing foam": "cleansing_foam",
    "foam cleanser": "cleansing_foam",
    "bars": "bars",
    "cloths & towelettes": "cloths_towelettes",
    "cloths and towelettes": "cloths_towelettes",

    "toner": "toner",
    "toners": "toner",
    "toners & astringents": "toners_astringents",
    "toners and astringents": "toners_astringents",
    "face mists": "face_mists",
    "mist": "mist",
    "mists": "mist",

    "serum": "serum",
    "serums": "serum",
    "essence": "essence",
    "ampoule": "ampoule",

    "cream": "cream",
    "creams": "cream",
    "moisturizer": "moisturizer",
    "moisturizers": "moisturizer",
    "creams & moisturizers": "creams_moisturizers",
    "creams and moisturizers": "creams_moisturizers",
    "face moisturizers": "face_moisturizers",
    "night creams": "night_creams",

    "face oil": "face_oil",
    "facial oil": "face_oil",
    "gel": "gel",
    "gels": "gel",

    "mask": "mask",
    "masks": "mask",
    "treatments & masks": "treatments_masks",
    "treatments and masks": "treatments_masks",

    "peel": "peel",
    "peels": "peel",
    "facial peels": "facial_peels",
    "polishes": "polishes",
    "scrubs": "scrub",
    "polishes & scrubs": "polishes_scrubs",
    "polishes and scrubs": "polishes_scrubs",
    "microdermabrasion": "microdermabrasion",

    "pore cleansing strips": "pore_cleansing_strips",
    "sets & kits": "sets_kits",
    "sets and kits": "sets_kits",
    "neck & d?collet?": "neck_decollete",
    "neck and decollete": "neck_decollete",
}

SKIN_TYPE_MAP = {
    "all": "all",
    "all skin": "all",
    "all skin type": "all",
    "all skin types": "all",
    "all types": "all",
    "any": "all",
    "normal": "normal",
    "normal skin": "normal",
    "dry": "dry",
    "dry skin": "dry",
    "oily": "oily",
    "oily skin": "oily",
    "combination": "combination",
    "combination skin": "combination",
    "sensitive": "sensitive",
    "sensitive skin": "sensitive",
    "acne prone": "acne_prone",
    "acne-prone": "acne_prone",
    "acne prone skin": "acne_prone",
    "acne prone skins": "acne_prone",
    "mature": None,
    "dehydrated": None,
    "acne": None,
    "rosacea": None,
}

INGREDIENT_MAP = {
    "hyaluronic acid": "hyaluronic_acid",
    "hyaluronic_acid": "hyaluronic_acid",
    "hyaluronicacid": "hyaluronic_acid",
    "niacinamide": "niacinamide",
    "retinol": "retinol",
    "retinyl palmitate": "retinol",
    "vitamin c": "vitamin_c",
    "vitamin_c": "vitamin_c",
    "vitaminc": "vitamin_c",
    "vitamin e": "vitamin_e",
    "vitamin_e": "vitamin_e",
    "vitamine": "vitamin_e",
    "tocopherol": "vitamin_e",
    "ceramide": "ceramide",
    "ceramides": "ceramide",
    "salicylic acid": "salicylic_acid",
    "salicylic_acid": "salicylic_acid",
    "glycolic acid": "glycolic_acid",
    "glycolic_acid": "glycolic_acid",
    "lactic acid": "lactic_acid",
    "lactic_acid": "lactic_acid",
    "peptide": "peptides",
    "peptides": "peptides",
    "aloe": "aloe",
    "aloe vera": "aloe_vera",
    "aloe vera extract": "aloe_vera",
    "centella": "centella",
    "centella asiatica": "centella",
    "centella asiatica extract": "centella",
    "cica": "centella",
    "propolis": "propolis",
    "propolis extract": "propolis",
    "snail": "snail",
    "snail mucin": "snail",
    "snail secretion filtrate": "snail",
    "glycerin": "glycerin",
    "squalane": "squalane",
    "green tea": "green_tea",
    "tea tree": "tea_tree",
    "tea tree oil": "tea_tree",
    "jojoba": "jojoba",
    "castor oil": "castor_oil",
    "panthenol": "panthenol",
    "collagen": "collagen",
}

BENEFIT_MAP = {
    "hydrating": "hydration",
    "hydrate": "hydration",
    "hydration": "hydration",
    "hydrating and soothing": "hydration",
    "hydration and soothing": "hydration",
    "moisturizing": "moisturizing",
    "moisturize": "moisturizing",
    "moisture": "moisturizing",
    "moisturizing and hydrating": "moisturizing",
    "soothing": "soothing",
    "soothe": "soothing",
    "calming": "soothing",
    "calm": "soothing",
    "cleansing": "cleansing",
    "cleanse": "cleansing",
    "exfoliating": "exfoliating",
    "exfoliate": "exfoliating",
    "brightening": "brightening",
    "brighten": "brightening",
    "anti aging": "antiaging",
    "anti-aging": "antiaging",
    "antiaging": "antiaging",
    "anti age": "antiaging",
    "anti wrinkle": "antiaging",
    "wrinkle reducing": "antiaging",
    "wrinkle-reducing": "antiaging",
    "firming": "firming",
    "firm": "firming",
    "nourishing": "nourishing",
    "nourish": "nourishing",
    "detoxifying": "detoxifying",
    "detox": "detoxifying",
    "rejuvenating": "rejuvenating",
    "rejuvenate": "rejuvenating",
    "antioxidant": "antioxidant",
    "barrier repair": "barrier_repair",
    "repair": "barrier_repair",
    "repairing": "barrier_repair",
}

SCENT_MAP = {
    "unscented": "unscented",
    "no scent": "unscented",
    "no fragrance": "unscented",
    "aloe": "aloe",
    "aloe vera": "aloe_vera",
    "fresh": "fresh",
    "rose": "rose",
    "tea tree": "tea_tree",
    "green tea": "green_tea",
    "coconut": "coconut",
    "lavender": "lavender",
    "citrus": "citrus",
    "orange": "orange",
    "peppermint": "peppermint",
    "honey": "honey",
    "pear": "pear",
    "natural": None,
}

FREE_CLAIM_MAP = {
    "fragrance free": "fragrance_free",
    "fragrance-free": "fragrance_free",
    "alcohol free": "alcohol_free",
    "alcohol-free": "alcohol_free",
    "paraben free": "paraben_free",
    "paraben-free": "paraben_free",
    "sulfate free": "sulfate_free",
    "sulfate-free": "sulfate_free",
    "oil free": "oil_free",
    "oil-free": "oil_free",
    "non comedogenic": "non_comedogenic",
    "non-comedogenic": "non_comedogenic",
    "cruelty free": "cruelty_free",
    "cruelty-free": "cruelty_free",
    "vegan": "vegan",
    "hypoallergenic": "hypoallergenic",
    "phthalate free": "phthalate_free",
    "phthalate-free": "phthalate_free",
    "dye free": "dye_free",
    "dye-free": "dye_free",
    "soap free": "soap_free",
    "soap-free": "soap_free",
    "gluten free": "gluten_free",
    "gluten-free": "gluten_free",
}

MARKETING_CLAIM_MAP = {
    "natural": "natural",
    "natural ingredients": "natural",
    "organic": "organic",
    "organic ingredients": "organic",
    "plant based": "plant_based",
    "plant-based": "plant_based",
    "dermatologist tested": "dermatologist_tested",
    "clinically tested": "clinically_tested",
    "eco friendly": "eco_friendly",
    "eco-friendly": "eco_friendly",
    "biodegradable": "biodegradable",
    "reusable": "reusable",
    "disposable": "disposable",
    "premium": "premium",
    "premium quality": "premium",
}

In [ ]:
# =========================================================
# Schema Helpers
# =========================================================
def clean_scalar_text_for_final(x):
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None
    text = str(x).strip()
    if not text:
        return None
    if text.lower() in NULL_LIKE_STRINGS:
        return None
    return text

def split_maybe_text_or_list(x):
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    if isinstance(x, (list, tuple, set, np.ndarray, pd.Series)):
        values = list(x)
    elif isinstance(x, str):
        stripped = x.strip()
        if not stripped or stripped.lower() in NULL_LIKE_STRINGS:
            return []
        if stripped.startswith("[") and stripped.endswith("]"):
            try:
                parsed = ast.literal_eval(stripped)
                values = list(parsed) if isinstance(parsed, (list, tuple, set)) else [stripped]
            except (ValueError, SyntaxError, TypeError, json.JSONDecodeError):
                try:
                    parsed = json.loads(stripped)
                    values = parsed if isinstance(parsed, list) else [stripped]
                except (ValueError, SyntaxError, TypeError, json.JSONDecodeError):
                    values = split_multi_value(stripped)
        else:
            values = split_multi_value(stripped)
    else:
        values = [x]
    out = []
    for value in values:
        cleaned = clean_scalar_text_for_final(value)
        if cleaned is not None:
            out.append(cleaned)
    return dedupe_keep_order(out)

def normalize_final_key(s):
    if s is None:
        return None
    s = str(s).strip().lower()
    s = s.replace("&", "and")
    s = " ".join(s.split())
    return s or None

def normalize_basic_token(x):
    text = clean_scalar_text_for_final(x)
    if text is None:
        return None
    text = text.lower().replace("&", " and ").replace("_", " ").replace("-", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else None

def canonicalize_sub_category(x):
    k = normalize_final_key(x)
    if k is None:
        return None
    return SUB_CATEGORY_CANONICAL_MAP.get(k, k.replace(" ", "_"))

def clean_list_field_for_final(x):
    return split_maybe_text_or_list(x)

def contains_null_like_value(x):
    if isinstance(x, list):
        return any((clean_text(value) or "").lower() in NULL_LIKE_STRINGS for value in x)
    text = clean_text(x)
    return text is not None and text.lower() in NULL_LIKE_STRINGS

def safe_join(vals):
    if isinstance(vals, list) and len(vals) > 0:
        return " | ".join(vals)
    return None

In [ ]:
# =========================================================
# Field Normalization
# =========================================================
def normalize_form(values):
    raw_vals = split_maybe_text_or_list(values)
    out = []
    for v in raw_vals:
        key = normalize_basic_token(v)
        if not key:
            continue
        mapped = FORM_MAP.get(key)
        if mapped:
            out.append(mapped)
        elif len(key) <= 30:
            out.append(key.replace(" ", "_"))
    return dedupe_keep_order(out)

def normalize_sub_category(values):
    raw_vals = sanitize_sub_category_values(values)
    out = []
    for v in raw_vals:
        cleaned_label = clean_sub_category_label(v)
        if cleaned_label is None:
            continue
        canonical = canonicalize_sub_category(cleaned_label)
        if canonical:
            out.append(canonical)
    return dedupe_keep_order(out)

def normalize_skin_type(values):
    raw_vals = split_maybe_text_or_list(values)
    skin_types = []
    for v in raw_vals:
        key = normalize_basic_token(v)
        if not key:
            continue
        mapped = SKIN_TYPE_MAP.get(key, "__MISSING__")
        if mapped != "__MISSING__":
            if mapped is not None:
                skin_types.append(mapped)
            continue
        if "acne" in key and "prone" in key:
            skin_types.append("acne_prone")
        elif key in {"dry", "oily", "combination", "sensitive", "normal", "all"}:
            skin_types.append(key)
    return dedupe_keep_order(skin_types), []

def normalize_ingredient(values):
    raw_vals = split_maybe_text_or_list(values)
    out = []
    for v in raw_vals:
        key = normalize_basic_token(v)
        if not key:
            continue
        mapped = INGREDIENT_MAP.get(key)
        if mapped:
            out.append(mapped)
        elif "hyaluronic" in key:
            out.append("hyaluronic_acid")
        elif "vitamin c" in key:
            out.append("vitamin_c")
        elif "vitamin e" in key or "tocopherol" in key:
            out.append("vitamin_e")
        elif "retinol" in key:
            out.append("retinol")
        elif "niacinamide" in key:
            out.append("niacinamide")
        elif "ceramide" in key:
            out.append("ceramide")
        elif "salicylic" in key:
            out.append("salicylic_acid")
        elif "glycolic" in key:
            out.append("glycolic_acid")
        elif "peptide" in key:
            out.append("peptides")
        elif len(key) <= 40:
            out.append(key.replace(" ", "_"))
    return dedupe_keep_order(out)

def normalize_benefit(values):
    raw_vals = split_maybe_text_or_list(values)
    out = []
    for v in raw_vals:
        key = normalize_basic_token(v)
        if not key:
            continue
        mapped = BENEFIT_MAP.get(key)
        if mapped:
            out.append(mapped)
        elif "hydrat" in key:
            out.append("hydration")
        elif "moistur" in key:
            out.append("moisturizing")
        elif "sooth" in key or "calm" in key:
            out.append("soothing")
        elif "clean" in key:
            out.append("cleansing")
        elif "exfol" in key:
            out.append("exfoliating")
        elif "bright" in key:
            out.append("brightening")
        elif "anti" in key and "aging" in key:
            out.append("antiaging")
        elif "firm" in key:
            out.append("firming")
        elif "barrier" in key or "repair" in key:
            out.append("barrier_repair")
        elif len(key) <= 40:
            out.append(key.replace(" ", "_"))
    return dedupe_keep_order(out)

def normalize_scent(values):
    raw_vals = split_maybe_text_or_list(values)
    scent_out = []
    reroute_free = []
    for v in raw_vals:
        key = normalize_basic_token(v)
        if not key:
            continue
        if key in {"fragrance free", "fragrance-free"}:
            reroute_free.append("fragrance_free")
            continue
        mapped = SCENT_MAP.get(key, "__MISSING__")
        if mapped != "__MISSING__":
            if mapped is not None:
                scent_out.append(mapped)
            continue
        if len(key) <= 30:
            scent_out.append(key.replace(" ", "_"))
    return dedupe_keep_order(scent_out), dedupe_keep_order(reroute_free)

def normalize_free_claim_and_marketing(values):
    raw_vals = split_maybe_text_or_list(values)
    free_out = []
    marketing_out = []
    for v in raw_vals:
        key = normalize_basic_token(v)
        if not key:
            continue
        free_mapped = FREE_CLAIM_MAP.get(key)
        if free_mapped:
            free_out.append(free_mapped)
            continue
        marketing_mapped = MARKETING_CLAIM_MAP.get(key)
        if marketing_mapped:
            marketing_out.append(marketing_mapped)
            continue
        if "free" in key:
            free_out.append(key.replace(" ", "_"))
        elif key in {"natural", "organic"}:
            marketing_out.append(key)
    return dedupe_keep_order(free_out), dedupe_keep_order(marketing_out)

In [ ]:
# =========================================================
# Item Schema Assembly
# =========================================================
norm_rows = []
for _, row in item_schema_base.iterrows():
    category_leaf_values = extract_sub_category_from_categories(row.get("categories_leaf", row.get("categories")))
    sub_category_raw = merge_sub_category_sources(split_maybe_text_or_list(row.get("sub_category_raw")), category_leaf_values)
    if not sub_category_raw:
        sub_category_raw = sanitize_sub_category_values(row.get("sub_category"))
        sub_category_raw = [clean_sub_category_label(v) for v in sub_category_raw]
        sub_category_raw = [v for v in sub_category_raw if v]
        sub_category_raw = dedupe_keep_order(sub_category_raw)
    form_norm = normalize_form(row.get("form"))
    sub_category_norm = normalize_sub_category(sub_category_raw)
    skin_type_norm, _ = normalize_skin_type(row.get("skin_type"))
    ingredient_norm = normalize_ingredient(row.get("ingredient"))
    benefit_norm = normalize_benefit(row.get("benefit"))
    scent_norm, reroute_free_from_scent = normalize_scent(row.get("scent"))
    free_claim_norm, marketing_claim_norm = normalize_free_claim_and_marketing(row.get("free_claim"))
    free_claim_norm = dedupe_keep_order(free_claim_norm + reroute_free_from_scent)
    concern_claim_norm = split_maybe_text_or_list(row.get("concern_claim_norm"))
    discontinued_raw = clean_scalar_text_for_final(row.get("is_discontinued_by_manufacturer_raw"))
    is_discontinued = bool(row.get("is_discontinued"))
    norm_rows.append({
        "parent_asin": row["parent_asin"],
        "sub_category_raw": sub_category_raw,
        "is_discontinued_by_manufacturer_raw": discontinued_raw,
        "is_discontinued": is_discontinued,
        "form_norm": form_norm,
        "sub_category_norm": sub_category_norm,
        "skin_type_norm": skin_type_norm,
        "ingredient_norm": ingredient_norm,
        "benefit_norm": benefit_norm,
        "scent_norm": scent_norm,
        "free_claim_norm": free_claim_norm,
        "marketing_claim_norm": marketing_claim_norm,
        "concern_claim_norm": concern_claim_norm,
        "sub_category_raw_text": safe_join(sub_category_raw),
        "form_norm_text": safe_join(form_norm),
        "sub_category_norm_text": safe_join(sub_category_norm),
        "skin_type_norm_text": safe_join(skin_type_norm),
        "ingredient_norm_text": safe_join(ingredient_norm),
        "benefit_norm_text": safe_join(benefit_norm),
        "scent_norm_text": safe_join(scent_norm),
        "free_claim_norm_text": safe_join(free_claim_norm),
        "marketing_claim_norm_text": safe_join(marketing_claim_norm),
        "concern_claim_norm_text": safe_join(concern_claim_norm),
    })
item_norm_df = pd.DataFrame(norm_rows)
drop_cols = [c for c in item_schema_base.columns if c in item_norm_df.columns and c != "parent_asin"]
item_schema_norm = item_schema_base.drop(columns=drop_cols, errors="ignore").merge(item_norm_df, on="parent_asin", how="left", validate="one_to_one")
item_schema_norm["parent_asin"] = item_schema_norm["parent_asin"].apply(clean_scalar_text_for_final)
if "brand_meta" in item_schema_norm.columns:
    item_schema_norm["brand_meta"] = item_schema_norm["brand_meta"].apply(clean_scalar_text_for_final)
if "is_discontinued_by_manufacturer_raw" in item_schema_norm.columns:
    item_schema_norm["is_discontinued_by_manufacturer_raw"] = item_schema_norm["is_discontinued_by_manufacturer_raw"].apply(clean_scalar_text_for_final)
if "is_discontinued" in item_schema_norm.columns:
    item_schema_norm["is_discontinued"] = item_schema_norm["is_discontinued"].fillna(False).astype(bool)
list_fields_for_final = ["sub_category", "sub_category_details", "sub_category_raw", "form_norm", "sub_category_norm", "skin_type_norm", "ingredient_norm", "benefit_norm", "scent_norm", "free_claim_norm", "marketing_claim_norm", "concern_claim_norm"]
for col in list_fields_for_final:
    if col in item_schema_norm.columns:
        item_schema_norm[col] = item_schema_norm[col].apply(clean_list_field_for_final)
text_fields_for_final = ["sub_category_text", "sub_category_raw_text", "form_norm_text", "sub_category_norm_text", "skin_type_norm_text", "ingredient_norm_text", "benefit_norm_text", "scent_norm_text", "free_claim_norm_text", "marketing_claim_norm_text", "concern_claim_norm_text"]
text_fields_for_final = [col for col in text_fields_for_final if col in item_schema_norm.columns]
for col in text_fields_for_final:
    item_schema_norm[col] = item_schema_norm[col].apply(clean_scalar_text_for_final)
rows_before_final_prune = len(item_schema_norm)
missing_parent_asin_mask = item_schema_norm["parent_asin"].isna()
key_structured_list_fields = ["form_norm", "sub_category_norm", "skin_type_norm", "ingredient_norm", "benefit_norm", "scent_norm", "free_claim_norm", "marketing_claim_norm", "concern_claim_norm"]
all_key_structured_fields_empty_mask = pd.Series(True, index=item_schema_norm.index)
if "brand_meta" in item_schema_norm.columns:
    all_key_structured_fields_empty_mask &= item_schema_norm["brand_meta"].isna()
for col in key_structured_list_fields:
    if col in item_schema_norm.columns:
        all_key_structured_fields_empty_mask &= item_schema_norm[col].apply(lambda x: isinstance(x, list) and len(x) == 0)
rows_all_structured_empty_before = int(all_key_structured_fields_empty_mask.sum())
drop_mask = missing_parent_asin_mask | all_key_structured_fields_empty_mask
drop_reason_summary_df = pd.DataFrame([
    {"drop_reason": "missing_parent_asin_only", "rows": int((missing_parent_asin_mask & ~all_key_structured_fields_empty_mask).sum())},
    {"drop_reason": "all_key_structured_fields_empty_only", "rows": int((~missing_parent_asin_mask & all_key_structured_fields_empty_mask).sum())},
    {"drop_reason": "both_reasons", "rows": int((missing_parent_asin_mask & all_key_structured_fields_empty_mask).sum())},
])
item_schema_norm = item_schema_norm.loc[~drop_mask].copy()
rows_after_final_prune = len(item_schema_norm)
rows_dropped_final_prune = rows_before_final_prune - rows_after_final_prune
post_all_key_structured_fields_empty_mask = pd.Series(True, index=item_schema_norm.index)
if "brand_meta" in item_schema_norm.columns:
    post_all_key_structured_fields_empty_mask &= item_schema_norm["brand_meta"].isna()
for col in key_structured_list_fields:
    if col in item_schema_norm.columns:
        post_all_key_structured_fields_empty_mask &= item_schema_norm[col].apply(lambda x: isinstance(x, list) and len(x) == 0)
rows_all_structured_empty_after = int(post_all_key_structured_fields_empty_mask.sum())
pass
pass
pass
pass
pass
pass
pass

In [ ]:
# =========================================================
# Quality Check
# =========================================================
coverage_rows = []
target_cols = ["brand_meta", "sub_category_text", "sub_category_raw_text", "sub_category_norm_text", "is_discontinued_by_manufacturer_raw", "form_norm_text", "skin_type_norm_text", "ingredient_norm_text", "benefit_norm_text", "scent_norm_text", "free_claim_norm_text", "marketing_claim_norm_text", "concern_claim_norm_text"]
for col in target_cols:
    if col in item_schema_norm.columns:
        non_null = item_schema_norm[col].notna().sum()
        coverage_rows.append({"column": col, "non_null_rows": int(non_null), "coverage_ratio": round(non_null / len(item_schema_norm), 6)})
if "is_discontinued" in item_schema_norm.columns:
    coverage_rows.append({"column": "is_discontinued_true", "non_null_rows": int(item_schema_norm["is_discontinued"].sum()), "coverage_ratio": round(item_schema_norm["is_discontinued"].mean(), 6)})
item_schema_coverage_df = pd.DataFrame(coverage_rows).sort_values(["coverage_ratio", "column"], ascending=[False, True])
list_fields_qc = ["sub_category_raw", "sub_category_norm", "form_norm", "skin_type_norm", "ingredient_norm", "benefit_norm", "scent_norm", "free_claim_norm", "marketing_claim_norm", "concern_claim_norm"]
list_fields_qc = [col for col in list_fields_qc if col in item_schema_norm.columns]
text_fields_qc = [col for col in target_cols if col in item_schema_norm.columns]
text_none_qc_df = pd.DataFrame([{"field": col, "none_count": int(item_schema_norm[col].isna().sum())} for col in text_fields_qc]).sort_values(["none_count", "field"], ascending=[False, True])
list_empty_qc_df = pd.DataFrame([{"field": col, "empty_list_count": int(item_schema_norm[col].apply(lambda x: isinstance(x, list) and len(x) == 0).sum())} for col in list_fields_qc]).sort_values(["empty_list_count", "field"], ascending=[False, True])
malformed_null_rows = []
for col in text_fields_qc:
    malformed_null_rows.append({"field": col, "malformed_null_like_count": int(item_schema_norm[col].apply(contains_null_like_value).sum())})
for col in list_fields_qc:
    malformed_null_rows.append({"field": col, "malformed_null_like_count": int(item_schema_norm[col].apply(contains_null_like_value).sum())})
malformed_null_qc_df = pd.DataFrame(malformed_null_rows).sort_values(["malformed_null_like_count", "field"], ascending=[False, True])
discontinued_qc_df = pd.DataFrame([
    {"metric": "discontinued_details_field_present", "value": int(item_schema_norm["is_discontinued_by_manufacturer_raw"].notna().sum()) if "is_discontinued_by_manufacturer_raw" in item_schema_norm.columns else 0},
    {"metric": "is_discontinued_true", "value": int(item_schema_norm["is_discontinued"].sum()) if "is_discontinued" in item_schema_norm.columns else 0},
    {"metric": "is_discontinued_false", "value": int((~item_schema_norm["is_discontinued"]).sum()) if "is_discontinued" in item_schema_norm.columns else 0},
])
discontinued_value_counts_df = pd.DataFrame([
    {"is_discontinued_by_manufacturer_raw": raw_value, "count": int(count)}
    for raw_value, count in item_schema_norm["is_discontinued_by_manufacturer_raw"].fillna("<missing>").value_counts(dropna=False).items()
]) if "is_discontinued_by_manufacturer_raw" in item_schema_norm.columns else pd.DataFrame(columns=["is_discontinued_by_manufacturer_raw", "count"])
structured_null_qc_df = pd.DataFrame([
    {"metric": "all_null_structured_rows_before_pruning", "value": rows_all_structured_empty_before},
    {"metric": "all_null_structured_rows_after_pruning", "value": rows_all_structured_empty_after},
    {"metric": "rows_dropped_in_final_pruning", "value": rows_dropped_final_prune},
])
pass
pass
pass
pass
pass
pass
pass
pass

In [ ]:
# =========================================================
# Top Value Audit
# =========================================================
top_value_rows = []
list_cols = ["sub_category_raw", "sub_category_norm", "form_norm", "skin_type_norm", "ingredient_norm", "benefit_norm", "scent_norm", "free_claim_norm", "marketing_claim_norm"]
for col in list_cols:
    if col in item_schema_norm.columns:
        counter = Counter()
        for vals in item_schema_norm[col]:
            if isinstance(vals, list):
                counter.update(vals)
        for rank, (val, cnt) in enumerate(counter.most_common(100), start=1):
            top_value_rows.append({"field": col, "rank": rank, "value": val, "count": cnt})
if "sub_category_norm_text" in item_schema_norm.columns:
    norm_text_counter = Counter(item_schema_norm["sub_category_norm_text"].dropna().astype(str))
    for rank, (val, cnt) in enumerate(norm_text_counter.most_common(100), start=1):
        top_value_rows.append({"field": "sub_category_norm_text", "rank": rank, "value": val, "count": cnt})
top_values_norm_df = pd.DataFrame(top_value_rows)
pass
pass
pass

In [ ]:
# =========================================================
# Historical Review-Derived Item Signals
# =========================================================
REVIEW_REPUTATION_ENABLED = True
RATING_USED_FOR_REVIEW_REPUTATION = False
SENTIMENT_USED_FOR_REVIEW_REPUTATION = False
LLM_USED_FOR_REVIEW_REPUTATION = False
RAW_REVIEW_TEXT_EXPORTED_FOR_REVIEW_REPUTATION = False
PARENT_ASIN_REQUIRED_NO_ASIN_FALLBACK_FOR_REPUTATION = True
HISTORICAL_REVIEWS_RESTRICTED_TO_CUTOFF = True
SAFE_REVIEW_REPUTATION_PREFIX_ALLOWED = True
UNSAFE_TARGET_REVIEW_COLUMNS_STILL_FORBIDDEN = True


def parquet_columns(path: Path):
    import pyarrow.parquet as pq
    return list(pq.ParquetFile(path).schema.names)


def normalize_review_space(value):
    if value is None or pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\n", " ").replace("\t", " ")).strip()


def to_review_timestamp_ms(series: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(series):
        parsed = pd.to_datetime(series, utc=True, errors="coerce")
        return (parsed.astype("int64") // 1_000_000).where(parsed.notna(), np.nan)

    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.dropna().empty:
        parsed = pd.to_datetime(series, utc=True, errors="coerce")
        return (parsed.astype("int64") // 1_000_000).where(parsed.notna(), np.nan)

    if numeric.dropna().max() < 10**11:
        numeric = numeric * 1000

    return numeric


# Load only the fields required to construct pre-cutoff review-derived item signals.
review_load_cols = ["parent_asin", "timestamp", "title", "text"]

review_columns = parquet_columns(REVIEWS_PATH)
missing_review_cols = sorted(set(review_load_cols) - set(review_columns))
if missing_review_cols:
    raise RuntimeError(f"Missing required review columns in REVIEWS_PATH: {missing_review_cols}")

raw_reviews_for_reputation = pd.read_parquet(
    REVIEWS_PATH,
    columns=review_load_cols,
)

raw_review_rows_loaded_for_reputation = int(len(raw_reviews_for_reputation))

review_reputation_df = pd.DataFrame({
    "parent_asin": raw_reviews_for_reputation["parent_asin"].astype(str).str.strip(),
    "review_timestamp_ms": to_review_timestamp_ms(raw_reviews_for_reputation["timestamp"]),
})

review_reputation_df["review_datetime"] = pd.to_datetime(
    review_reputation_df["review_timestamp_ms"],
    unit="ms",
    utc=True,
    errors="coerce",
)

title_source = raw_reviews_for_reputation["title"].map(normalize_review_space)
text_source = raw_reviews_for_reputation["text"].map(normalize_review_space)

review_reputation_df["review_reputation_source_text"] = (
    title_source + " " + text_source
).map(normalize_review_space)

review_reputation_df = review_reputation_df[
    review_reputation_df["parent_asin"].ne("")
    & review_reputation_df["review_datetime"].notna()
].copy()

historical_review_mask = review_reputation_df["review_datetime"] < TRAIN_REVIEW_CUTOFF_EXCLUSIVE
historical_reviews_for_reputation = review_reputation_df.loc[historical_review_mask].copy()
reviews_excluded_by_reputation_cutoff = int((~historical_review_mask).sum())
catalog_overlap_item_count = int(
    historical_reviews_for_reputation["parent_asin"].nunique()
)

print("Catalog overlap items:", catalog_overlap_item_count)
if catalog_overlap_item_count <= 0:
    raise RuntimeError(
        "No historical review items overlap with the catalog."
    )

In [ ]:
REVIEW_REPUTATION_SIGNAL_PATTERNS = {
    "review_reputation_concern": {
        "acne": [r"\bacne\b", r"\bbreakouts?\b", r"\bblemishes?\b", r"\bpimples?\b"],
        "redness": [r"\bredness\b", r"\bred skin\b"],
        "dark spots": [r"\bdark spots?\b", r"\bhyperpigmentation\b", r"\buneven tone\b"],
        "pores": [r"\bpores?\b", r"\blarge pores?\b"],
        "dryness": [r"\bdry(ness)?\b", r"\bdehydrat(ed|ion)\b", r"\bflaky\b"],
        "oiliness": [r"\boily\b", r"\bgreasy\b", r"\bshine\b"],
        "dullness": [r"\bdull(ness)?\b"],
        "fine lines": [r"\bfine lines?\b", r"\bwrinkles?\b"],
        "texture": [r"\btexture\b", r"\brough\b", r"\bbumpy\b"],
    },
    "review_reputation_skin_type": {
        "dry skin": [r"\bdry skin\b"],
        "oily skin": [r"\boily skin\b"],
        "combination skin": [r"\bcombination skin\b", r"\bcombo skin\b"],
        "sensitive skin": [r"\bsensitive skin\b", r"\bsensitiv(e|ity)\b"],
        "acne-prone skin": [r"\bacne prone\b", r"\bacne-prone\b"],
    },
    "review_reputation_benefit": {
        "hydrating": [r"\bhydrat\w*", r"\bmoisturi[sz]\w*"],
        "soothing": [r"\bsooth\w*", r"\bcalm\w*"],
        "brightening": [r"\bbrighten\w*", r"\bglow\w*"],
        "smoothing": [r"\bsmooth\w*", r"\bsoften\w*"],
        "firming": [r"\bfirm\w*", r"\btighten\w*"],
        "cleansing": [r"\bcleanse\w*", r"\bremoves? makeup\b"],
        "exfoliating": [r"\bexfoliat\w*", r"\bpeel\b"],
    },
    "review_reputation_ingredient": {
        "hyaluronic acid": [r"\bhyaluronic acid\b"],
        "niacinamide": [r"\bniacinamide\b"],
        "retinol": [r"\bretinol\b", r"\bretinoid\b"],
        "vitamin c": [r"\bvitamin c\b"],
        "ceramide": [r"\bceramides?\b"],
        "salicylic acid": [r"\bsalicylic acid\b", r"\bbha\b"],
        "glycolic acid": [r"\bglycolic acid\b", r"\baha\b"],
        "centella": [r"\bcentella\b", r"\bcica\b"],
        "aloe": [r"\baloe\b"],
        "tea tree": [r"\btea tree\b"],
    },
    "review_reputation_product_type_or_form_texture": {
        "cleanser": [r"\bcleanser\b", r"\bface wash\b"],
        "serum": [r"\bserum\b"],
        "moisturizer": [r"\bmoisturi[sz]er\b", r"\bcream\b", r"\blotion\b"],
        "toner": [r"\btoner\b"],
        "mask": [r"\bmask\b"],
        "sunscreen": [r"\bsunscreen\b", r"\bspf\b"],
        "balm": [r"\bbalm\b"],
        "oil": [r"\boil\b"],
        "gel": [r"\bgel\b"],
        "foam": [r"\bfoam\w*\b"],
        "lightweight": [r"\blight ?weight\b"],
        "rich texture": [r"\brich\b", r"\bthick\b"],
        "non-greasy": [r"\bnon greasy\b", r"\bnon-greasy\b"],
    },
}
REVIEW_REPUTATION_FAMILY_LABELS = {
    "review_reputation_concern": "review concerns",
    "review_reputation_skin_type": "review skin types",
    "review_reputation_benefit": "review benefits",
    "review_reputation_ingredient": "review ingredients",
    "review_reputation_product_type_or_form_texture": "review product forms",
}
REVIEW_REPUTATION_MAX_PHRASES_PER_FAMILY = 12


def extract_review_reputation_signals(text):
    lowered = normalize_review_space(text).lower()
    out = {}
    for family, signal_patterns in REVIEW_REPUTATION_SIGNAL_PATTERNS.items():
        hits = []
        for label, patterns in signal_patterns.items():
            if any(re.search(pattern, lowered) for pattern in patterns):
                hits.append(label)
        out[family] = hits
    return out


family_signal_counters = {family: Counter() for family in REVIEW_REPUTATION_SIGNAL_PATTERNS}
item_signal_counters = defaultdict(lambda: {family: Counter() for family in REVIEW_REPUTATION_SIGNAL_PATTERNS})
item_review_counts = historical_reviews_for_reputation.groupby("parent_asin").size().astype(int)

for parent_asin, text in historical_reviews_for_reputation[["parent_asin", "review_reputation_source_text"]].itertuples(index=False):
    signals = extract_review_reputation_signals(text)
    for family, labels in signals.items():
        unique_labels = sorted(set(labels))
        family_signal_counters[family].update(unique_labels)
        item_signal_counters[parent_asin][family].update(unique_labels)

reputation_rows = []
for parent_asin, family_counters in item_signal_counters.items():
    row = {
        "parent_asin": parent_asin,
        "historical_review_count_pre_window": int(item_review_counts.get(parent_asin, 0)),
    }
    reputation_text_parts = []
    signal_family_count = 0
    signal_total_count = 0
    for family, label in REVIEW_REPUTATION_FAMILY_LABELS.items():
        top_labels = [name for name, _ in family_counters[family].most_common(REVIEW_REPUTATION_MAX_PHRASES_PER_FAMILY)]
        text_value = ", ".join(top_labels)
        row[f"{family}_text"] = text_value
        if top_labels:
            signal_family_count += 1
            signal_total_count += len(top_labels)
            reputation_text_parts.append(f"{label}: {text_value}")
    row["historical_review_reputation_text"] = " | ".join(reputation_text_parts)
    row["review_reputation_signal_family_count"] = int(signal_family_count)
    row["review_reputation_signal_total_count"] = int(signal_total_count)
    row["review_reputation_available_flag"] = bool(signal_total_count > 0)
    reputation_rows.append(row)

review_reputation_features_df = pd.DataFrame(reputation_rows)
review_reputation_cols = [
    "historical_review_count_pre_window",
    "historical_review_reputation_text",
    "review_reputation_concern_text",
    "review_reputation_skin_type_text",
    "review_reputation_benefit_text",
    "review_reputation_ingredient_text",
    "review_reputation_product_type_or_form_texture_text",
    "review_reputation_signal_family_count",
    "review_reputation_signal_total_count",
    "review_reputation_available_flag",
]
if review_reputation_features_df.empty:
    review_reputation_features_df = pd.DataFrame(columns=["parent_asin"] + review_reputation_cols)

item_schema_norm = item_schema_norm.merge(review_reputation_features_df, on="parent_asin", how="left", validate="one_to_one")
text_reputation_cols = [
    "historical_review_reputation_text",
    "review_reputation_concern_text",
    "review_reputation_skin_type_text",
    "review_reputation_benefit_text",
    "review_reputation_ingredient_text",
    "review_reputation_product_type_or_form_texture_text",
]
for col in text_reputation_cols:
    item_schema_norm[col] = item_schema_norm[col].fillna("").map(normalize_review_space)
for col in ["historical_review_count_pre_window", "review_reputation_signal_family_count", "review_reputation_signal_total_count"]:
    item_schema_norm[col] = item_schema_norm[col].fillna(0).astype(int)
item_schema_norm["review_reputation_available_flag"] = item_schema_norm["review_reputation_available_flag"].fillna(False).astype(bool)
item_schema_norm["review_reputation_facet_text"] = item_schema_norm["historical_review_reputation_text"]
item_schema_norm["review_reputation_source"] = REVIEW_REPUTATION_SOURCE
item_schema_norm["review_reputation_cutoff_exclusive"] = TRAIN_REVIEW_CUTOFF_EXCLUSIVE.isoformat()


if historical_reviews_for_reputation["review_datetime"].ge(TRAIN_REVIEW_CUTOFF_EXCLUSIVE).any():
    raise RuntimeError("Review reputation source contains reviews on or after TRAIN_REVIEW_CUTOFF_EXCLUSIVE.")
for forbidden_raw_col in ["review_reputation_source_text", "review_text", "target_review_text", "heldout_review_text"]:
    if forbidden_raw_col in item_schema_norm.columns:
        raise RuntimeError(f"Raw review text column exported to item schema: {forbidden_raw_col}")

items_with_reputation = int(item_schema_norm["review_reputation_available_flag"].sum())
items_without_reputation = int((~item_schema_norm["review_reputation_available_flag"]).sum())
review_reputation_diag_rows = [
    {"metric": "total_raw_reviews_loaded", "family": "overall", "value": raw_review_rows_loaded_for_reputation},
    {"metric": "historical_training_reviews_before_cutoff", "family": "overall", "value": int(len(historical_reviews_for_reputation))},
    {"metric": "reviews_excluded_by_cutoff", "family": "overall", "value": reviews_excluded_by_reputation_cutoff},
    {"metric": "items_with_historical_reputation", "family": "overall", "value": items_with_reputation},
    {"metric": "items_without_historical_reputation", "family": "overall", "value": items_without_reputation},
    {"metric": "average_historical_review_count_pre_window", "family": "overall", "value": float(item_schema_norm["historical_review_count_pre_window"].mean())},
]
for family in REVIEW_REPUTATION_SIGNAL_PATTERNS:
    text_col = f"{family}_text"
    non_empty_rate = float(item_schema_norm[text_col].ne("").mean()) if text_col in item_schema_norm.columns else 0.0
    review_reputation_diag_rows.append({"metric": "non_empty_rate", "family": family, "value": non_empty_rate})
    for rank, (signal, count) in enumerate(family_signal_counters[family].most_common(20), start=1):
        review_reputation_diag_rows.append({"metric": f"top_signal_{rank:02d}:{signal}", "family": family, "value": int(count)})
review_reputation_diagnostics_df = pd.DataFrame(review_reputation_diag_rows)
review_reputation_diagnostics_df.to_csv(OUTPUT_DIR / "face_item_review_reputation_diagnostics.csv", index=False, encoding="utf-8-sig")

del raw_reviews_for_reputation
del review_reputation_df
del historical_reviews_for_reputation

pass
pass
pass
pass

## Harmonized facet and retrieval-evidence policy

This section keeps brand, product-functional facets, and historical review-derived
item evidence in separate channels. It creates three retrieval views:

- **Core:** catalog metadata, the separate brand facet, and product-functional facets.
- **Review-only:** normalized item-level signals extracted from reviews before the
  frozen training cutoff.
- **Global Review:** catalog metadata, the separate brand facet, product-functional facets, and historical
  review-derived item evidence before the frozen training cutoff. The production
  columns are `canonical_retrieval_text`, `canonical_text_dense`, and
  `canonical_text_sparse`.


In [ ]:
# =========================================================
# Harmonized Global Review Evidence Contract
# =========================================================
FACET_POLICY_VERSION = "harmonized_v2_global_review_brand_retrieval_profile"
PRODUCTION_EVIDENCE_SCOPE = "catalog_metadata_functional_facets_and_historical_review_signals"
PRODUCTION_HISTORICAL_REVIEW_REPUTATION_ENABLED = True
HISTORICAL_REVIEW_REPUTATION_QUARANTINED = False
BRAND_POLICY = "separate_preference_facet__query_unsafe__retrieval_profile_graph_safe"


def _harm_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(value, (list, tuple, set, np.ndarray, pd.Series)):
        return " | ".join(_harm_text(v) for v in value if _harm_text(v))
    if isinstance(value, dict):
        return " | ".join(_harm_text(v) for v in value.values() if _harm_text(v))
    text = re.sub(r"\s+", " ", str(value).replace("\n", " ").replace("\t", " ")).strip()
    if text.lower() in {"", "none", "null", "nan", "n/a", "na", "[]", "{}"}:
        return ""
    return text


def _harm_split(value):
    text = _harm_text(value)
    if not text:
        return []
    pieces = []
    for part in re.split(r"\s*\|\s*|\s*;\s*", text):
        part = _harm_text(part)
        if part:
            pieces.append(part)
    return pieces


def _harm_join(values):
    out = []
    seen = set()
    for value in values:
        for part in _harm_split(value):
            key = part.lower()
            if key and key not in seen:
                seen.add(key)
                out.append(part)
    return " | ".join(out)


def _harm_join_cols(row, cols):
    return _harm_join(row[col] for col in cols if col in row.index)


HARMONIZED_BRAND_SOURCE_COLS = ["brand_meta"]
HARMONIZED_QUERY_SAFE_FACET_SOURCES = {
    "facet_category_text": ["sub_category_norm_text", "sub_category_text"],
    "facet_form_text": ["form_norm_text"],
    "facet_skin_type_text": ["skin_type_norm_text"],
    "facet_ingredient_text": ["ingredient_norm_text"],
    "facet_benefit_text": ["benefit_norm_text", "concern_claim_norm_text"],
    "facet_claim_text": ["free_claim_norm_text", "marketing_claim_norm_text"],
    "facet_scent_text": ["scent_norm_text"],
}
HARMONIZED_REVIEW_REPUTATION_COLS = [
    "historical_review_reputation_text",
    "review_reputation_concern_text",
    "review_reputation_skin_type_text",
    "review_reputation_benefit_text",
    "review_reputation_ingredient_text",
    "review_reputation_product_type_or_form_texture_text",
]
HARMONIZED_IDENTIFIER_DIAGNOSTIC_COLS = [
    "asin", "parent_asin_raw", "model", "item_model_number", "upc", "ean",
    "package_dimensions", "unit_count", "number_of_items", "size", "exact_dosage",
]
FACE_GENERIC_CATEGORY_ANCHORS = {
    "skin care",
    "skincare",
    "face",
    "facial",
    "skin care routine",
    "skincare routine",
}
FACE_GENERIC_UTILITY_TOKENS = {
    "support",
    "supports",
    "help",
    "helps",
    "promote",
    "promotes",
    "boost",
    "formula",
    "blend",
    "complex",
    "product",
    "solution",
}
FACE_CONTEXT_DEPENDENT_UTILITY_TOKENS = {
    "daily",
    "natural",
    "wellness",
    "health",
    "care",
    "routine",
}
FACE_SPECIFIC_PHRASE_EXCEPTIONS = {
    "morning routine",
    "night routine",
    "skin barrier support",
    "hydration support",
    "sensitive skin care",
    "acne care",
}

query_safe_cols = list(HARMONIZED_QUERY_SAFE_FACET_SOURCES.keys())
QUERY_SAFE_FACET_SOURCE_COLUMNS = [
    source_col
    for source_cols in HARMONIZED_QUERY_SAFE_FACET_SOURCES.values()
    for source_col in source_cols
]
FUNCTIONAL_FACET_SOURCE_COLUMNS = ["specific_query_safe_facet_text"]
CORE_TEXT_SOURCE_COLUMNS = ["title", "brand_facet_text", "query_safe_facet_text", "description", "features"]
CORE_SPARSE_TEXT_SOURCE_COLUMNS = ["title", "brand_facet_text", "query_safe_facet_text", "features"]
# Brand is retained as a separate profile source and is excluded only from synthetic queries.
PROFILE_SOURCE_TEXT_CORE_SOURCE_COLUMNS = [
    "brand_facet_text",
    "functional_facet_text",
    "title",
    "description",
]
HISTORICAL_REVIEW_TEXT_COLUMNS = HARMONIZED_REVIEW_REPUTATION_COLS + [
    "review_reputation_facet_text",
    *HARMONIZED_REVIEW_REPUTATION_COLS,
    "review_reputation_facet_text",
    *HARMONIZED_REVIEW_REPUTATION_COLS,
    "review_reputation_facet_text",
    "review_reputation_only_text",
    "review_reputation_available_flag",
    "review_reputation_signal_family_count",
    "review_reputation_signal_total_count",
    "review_reputation_available_flag",
    "review_reputation_signal_family_count",
    "review_reputation_signal_total_count",
]
SOURCE_LISTS_USED_FOR_PRODUCTION_CORE = {
    "query_safe_facet_text": QUERY_SAFE_FACET_SOURCE_COLUMNS,
    "functional_facet_text": FUNCTIONAL_FACET_SOURCE_COLUMNS,
    "canonical_retrieval_text_core": CORE_TEXT_SOURCE_COLUMNS,
    "canonical_text_dense_core": CORE_TEXT_SOURCE_COLUMNS,
    "canonical_text_sparse_core": CORE_SPARSE_TEXT_SOURCE_COLUMNS,
    "profile_source_text_core": PROFILE_SOURCE_TEXT_CORE_SOURCE_COLUMNS,
}

item_schema_norm["facet_brand_text"] = item_schema_norm.apply(
    lambda row: _harm_join_cols(row, HARMONIZED_BRAND_SOURCE_COLS),
    axis=1,
)
item_schema_norm["brand_facet_text"] = item_schema_norm["facet_brand_text"]

for out_col, source_cols in HARMONIZED_QUERY_SAFE_FACET_SOURCES.items():
    item_schema_norm[out_col] = item_schema_norm.apply(
        lambda row, cols=source_cols: _harm_join_cols(row, cols),
        axis=1,
    )

def _split_facet_values(value):
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass

    values = []
    for part in re.split(r"\s*\|\s*|\s*;\s*|\s*,\s*", str(value)):
        cleaned = _harm_text(part)
        if cleaned:
            values.append(cleaned)

    return values


FACE_BAD_BRAND_FACET_VALUES = (
    FACE_GENERIC_CATEGORY_ANCHORS
    | FACE_GENERIC_UTILITY_TOKENS
    | FACE_CONTEXT_DEPENDENT_UTILITY_TOKENS
)


def _remove_bad_brand_values(value):
    kept = [
        part
        for part in _split_facet_values(value)
        if part.lower() not in FACE_BAD_BRAND_FACET_VALUES
    ]
    return _harm_join(kept)


item_schema_norm["facet_brand_text"] = (
    item_schema_norm["facet_brand_text"].map(_remove_bad_brand_values)
)
item_schema_norm["brand_facet_text"] = item_schema_norm["facet_brand_text"]

face_bad_brand_norms = {value.lower() for value in FACE_BAD_BRAND_FACET_VALUES}
face_bad_brand_rows = item_schema_norm[
    item_schema_norm["brand_facet_text"]
    .fillna("")
    .astype(str)
    .map(lambda value: bool(
        {part.lower() for part in _split_facet_values(value)}
        & face_bad_brand_norms
    ))
]

if len(face_bad_brand_rows):
    display(face_bad_brand_rows[["parent_asin", "brand_facet_text"]].head(20))
    raise RuntimeError("Generic/category/utility tokens leaked into brand_facet_text.")


def _remove_row_brand_values(text_value, brand_value):
    brand_norms = {
        value.lower()
        for value in _split_facet_values(brand_value)
    }

    kept = [
        value
        for value in _split_facet_values(text_value)
        if value.lower() not in brand_norms
    ]

    return _harm_join(kept)


for col in query_safe_cols:
    item_schema_norm[col] = [
        _remove_row_brand_values(text_value, brand_value)
        for text_value, brand_value in zip(
            item_schema_norm[col],
            item_schema_norm["brand_facet_text"],
        )
    ]

item_schema_norm["query_safe_facet_text"] = item_schema_norm.apply(
    lambda row: _harm_join_cols(row, query_safe_cols),
    axis=1,
)

_generic_anchor_norms = {value.lower() for value in FACE_GENERIC_CATEGORY_ANCHORS}
_generic_utility_norms = {value.lower() for value in FACE_GENERIC_UTILITY_TOKENS}
_context_utility_norms = {value.lower() for value in FACE_CONTEXT_DEPENDENT_UTILITY_TOKENS}
_specific_exception_norms = {value.lower() for value in FACE_SPECIFIC_PHRASE_EXCEPTIONS}


def _partition_query_safe_text(value):
    generic_anchors = []
    generic_utilities = []
    context_utilities = []
    specific_values = []

    for part in _harm_split(value):
        norm = part.lower()
        if norm in _specific_exception_norms:
            specific_values.append(part)
        elif norm in _generic_anchor_norms:
            generic_anchors.append(part)
        elif norm in _generic_utility_norms:
            generic_utilities.append(part)
        elif norm in _context_utility_norms:
            context_utilities.append(part)
        else:
            specific_values.append(part)

    return pd.Series({
        "generic_category_anchor_text": _harm_join(generic_anchors),
        "generic_utility_token_text": _harm_join(generic_utilities),
        "context_dependent_utility_token_text": _harm_join(context_utilities),
        "specific_query_safe_facet_text": _harm_join(specific_values),
    })


partition_columns = [
    "generic_category_anchor_text",
    "generic_utility_token_text",
    "context_dependent_utility_token_text",
    "specific_query_safe_facet_text",
]
item_schema_norm = item_schema_norm.drop(columns=partition_columns, errors="ignore")
item_schema_norm = pd.concat(
    [item_schema_norm, item_schema_norm["query_safe_facet_text"].apply(_partition_query_safe_text)],
    axis=1,
)
item_schema_norm["functional_facet_text"] = item_schema_norm["specific_query_safe_facet_text"]


def _normalized_facet_values(value):
    return {
        part.lower()
        for part in _split_facet_values(value)
    }


brand_in_query_safe_source_rows = int(
    item_schema_norm.apply(
        lambda row: any(
            bool(
                _normalized_facet_values(row.get(col))
                & _normalized_facet_values(row.get("brand_facet_text"))
            )
            for col in query_safe_cols
        ),
        axis=1,
    ).sum()
)
if brand_in_query_safe_source_rows:
    raise RuntimeError(
        "Query-safe facet source columns still contain row brand values: "
        f"{brand_in_query_safe_source_rows}"
    )

brand_in_functional_rows = int(
    item_schema_norm.apply(
        lambda row: bool(
            _normalized_facet_values(row.get("functional_facet_text"))
            & _normalized_facet_values(row.get("brand_facet_text"))
        ),
        axis=1,
    ).sum()
)
if brand_in_functional_rows:
    raise RuntimeError(
        "functional_facet_text still contains row brand values: "
        f"{brand_in_functional_rows}"
    )

item_schema_norm["review_reputation_facet_text"] = item_schema_norm.apply(
    lambda row: _harm_join_cols(row, HARMONIZED_REVIEW_REPUTATION_COLS),
    axis=1,
)
item_schema_norm["identifier_diagnostic_text"] = item_schema_norm.apply(
    lambda row: _harm_join_cols(row, HARMONIZED_IDENTIFIER_DIAGNOSTIC_COLS),
    axis=1,
)

item_schema_norm["canonical_metadata_text"] = item_schema_norm.apply(
    lambda row: _harm_join([
        row.get("title"),
        row.get("brand_facet_text"),
        row.get("query_safe_facet_text"),
        row.get("description"),
        row.get("features"),
    ]),
    axis=1,
)
item_schema_norm["canonical_item_metadata_source_text"] = item_schema_norm["canonical_metadata_text"]

item_schema_norm["canonical_retrieval_text_core"] = item_schema_norm.apply(
    lambda row: _harm_join([row.get(col) for col in CORE_TEXT_SOURCE_COLUMNS]),
    axis=1,
)
item_schema_norm["canonical_text_dense_core"] = item_schema_norm["canonical_retrieval_text_core"]
item_schema_norm["canonical_text_sparse_core"] = item_schema_norm.apply(
    lambda row: _harm_join([row.get(col) for col in CORE_SPARSE_TEXT_SOURCE_COLUMNS]),
    axis=1,
)

item_schema_norm["review_reputation_only_text"] = (
    item_schema_norm["review_reputation_facet_text"]
    .fillna("")
    .astype(str)
    .map(_harm_text)
)

item_schema_norm["canonical_retrieval_text"] = item_schema_norm.apply(
    lambda row: _harm_join([
        row.get("canonical_retrieval_text_core"),
        row.get("review_reputation_only_text"),
    ]),
    axis=1,
)
item_schema_norm["canonical_text_dense"] = item_schema_norm["canonical_retrieval_text"]
item_schema_norm["canonical_text_sparse"] = item_schema_norm.apply(
    lambda row: _harm_join([
        row.get("canonical_text_sparse_core"),
        row.get("review_reputation_only_text"),
    ]),
    axis=1,
)

item_schema_norm["profile_safe_facet_text"] = item_schema_norm.apply(
    lambda row: _harm_join([
        row.get("brand_facet_text"),
        row.get("functional_facet_text"),
    ]),
    axis=1,
)
item_schema_norm["profile_source_text_core"] = item_schema_norm.apply(
    lambda row: _harm_join([row.get(col) for col in PROFILE_SOURCE_TEXT_CORE_SOURCE_COLUMNS]),
    axis=1,
)
item_schema_norm["profile_source_text_dedup_seed"] = item_schema_norm["profile_source_text_core"]

brand_rows_mask = item_schema_norm["brand_facet_text"].fillna("").astype(str).str.strip().ne("")
brand_missing_retrieval_rows = int(
    item_schema_norm.loc[brand_rows_mask].apply(
        lambda row: not _normalized_facet_values(row.get("brand_facet_text")).issubset(
            _normalized_facet_values(row.get("canonical_retrieval_text_core"))
        ),
        axis=1,
    ).sum()
)
brand_missing_profile_rows = int(
    item_schema_norm.loc[brand_rows_mask].apply(
        lambda row: not _normalized_facet_values(row.get("brand_facet_text")).issubset(
            _normalized_facet_values(row.get("profile_safe_facet_text"))
        ),
        axis=1,
    ).sum()
)
if brand_missing_retrieval_rows:
    raise RuntimeError(
        f"Brand values are missing from retrieval text for {brand_missing_retrieval_rows} items."
    )
if brand_missing_profile_rows:
    raise RuntimeError(
        f"Brand values are missing from profile-safe facets for {brand_missing_profile_rows} items."
    )

item_schema_norm["facet_policy_version"] = FACET_POLICY_VERSION
item_schema_norm["evidence_scope"] = PRODUCTION_EVIDENCE_SCOPE
item_schema_norm["historical_review_reputation_enabled"] = PRODUCTION_HISTORICAL_REVIEW_REPUTATION_ENABLED
item_schema_norm["historical_review_reputation_quarantined"] = HISTORICAL_REVIEW_REPUTATION_QUARANTINED
item_schema_norm["brand_policy"] = BRAND_POLICY
item_schema_norm["brand_retrieval_enabled"] = True
item_schema_norm["brand_profile_enabled"] = True
item_schema_norm["brand_query_enabled"] = False
item_schema_norm["identifier_policy"] = "diagnostic_only__excluded_from_query_safe_profile_and_canonical_retrieval_text"

if item_schema_norm["query_safe_facet_text"].map(lambda x: isinstance(x, str)).sum() != len(item_schema_norm):
    raise RuntimeError("query_safe_facet_text must be a string for every item.")
if item_schema_norm["brand_facet_text"].map(lambda x: isinstance(x, str)).sum() != len(item_schema_norm):
    raise RuntimeError("brand_facet_text must be a string for every item.")
if item_schema_norm["canonical_retrieval_text"].map(lambda x: isinstance(x, str)).sum() != len(item_schema_norm):
    raise RuntimeError("canonical_retrieval_text must be a string for every item.")

facet_policy_rows = []
for col in ["brand_facet_text", "facet_brand_text"] + query_safe_cols + [
    "query_safe_facet_text",
    "specific_query_safe_facet_text",
    "functional_facet_text",
    "review_reputation_facet_text",
    "identifier_diagnostic_text",
    "canonical_metadata_text",
    "canonical_item_metadata_source_text",
    "canonical_retrieval_text_core",
    "canonical_text_dense_core",
    "canonical_text_sparse_core",
    "canonical_retrieval_text",
    "canonical_text_dense",
    "canonical_text_sparse",
    "profile_safe_facet_text",
    "profile_source_text_core",
    "profile_source_text_dedup_seed",
]:
    facet_policy_rows.append({
        "column": col,
        "non_empty_rows": int(item_schema_norm[col].map(lambda x: _harm_text(x) != "").sum()),
        "coverage_ratio": float(item_schema_norm[col].map(lambda x: _harm_text(x) != "").mean()) if len(item_schema_norm) else 0.0,
        "policy": (
            "brand_retrieval_profile_safe_not_query_safe" if col in {"brand_facet_text", "facet_brand_text", "profile_safe_facet_text"} else
            "specific_product_functional_query_safe" if col in {"specific_query_safe_facet_text", "functional_facet_text"} else
            "query_safe_nonbrand_nonidentifier" if col in query_safe_cols or col == "query_safe_facet_text" else
            "global_train_only_review_reputation" if col == "review_reputation_facet_text" else
            "diagnostic_only_not_for_query_or_retrieval" if col == "identifier_diagnostic_text" else
            "production_global_review_text" if col in {"canonical_retrieval_text", "canonical_text_dense", "canonical_text_sparse"} else
            "catalog_metadata_functional_facets_and_historical_review_signals"
        ),
    })
facet_policy_summary_df = pd.DataFrame(facet_policy_rows)
facet_policy_summary_df.to_csv(OUTPUT_DIR / "face_item_schema_facet_policy_summary.csv", index=False, encoding="utf-8-sig")

facet_policy_manifest = {
    "facet_policy_version": FACET_POLICY_VERSION,
    "category": "facial_skincare",
    "evidence_scope": PRODUCTION_EVIDENCE_SCOPE,
    "historical_review_reputation_enabled": PRODUCTION_HISTORICAL_REVIEW_REPUTATION_ENABLED,
    "historical_review_reputation_quarantined": HISTORICAL_REVIEW_REPUTATION_QUARANTINED,
    "brand_source_columns": HARMONIZED_BRAND_SOURCE_COLS,
    "query_safe_facet_sources": HARMONIZED_QUERY_SAFE_FACET_SOURCES,
    "review_reputation_columns": HARMONIZED_REVIEW_REPUTATION_COLS,
    "identifier_diagnostic_columns": HARMONIZED_IDENTIFIER_DIAGNOSTIC_COLS,
    "core_text_source_columns": CORE_TEXT_SOURCE_COLUMNS,
    "core_sparse_text_source_columns": CORE_SPARSE_TEXT_SOURCE_COLUMNS,
    "profile_source_text_core_source_columns": PROFILE_SOURCE_TEXT_CORE_SOURCE_COLUMNS,
    "global_train_cutoff_exclusive": TRAIN_REVIEW_CUTOFF_EXCLUSIVE.isoformat(),
    "brand_in_query_safe_text": False,
    "brand_in_retrieval_text": True,
    "brand_in_profile_source_text": True,
    "brand_is_product_functional_facet": False,
    "identifiers_in_query_safe_text": False,
    "historical_reviews_in_core_text": False,
    "historical_reviews_in_production_retrieval_text": True,
    "historical_reviews_in_profile_source_text": False,
    "production_text_columns": [
        "canonical_retrieval_text",
        "canonical_text_dense",
        "canonical_text_sparse",
    ],
    "same_item_prior_policy_for_09": "outside Notebook 02; user-specific prior enters from Notebook 08 onward",
    "profile_policy_for_09": "brand_and_product_functional_facets_are_profile_safe; historical review signals remain item-side retrieval evidence",
    "downstream_retrieval_text_column_for_03": "canonical_retrieval_text",
    "downstream_profile_seed_column_for_09": "profile_source_text_dedup_seed",
}
with open(OUTPUT_DIR / "face_feature_engineering_facet_policy_manifest.json", "w", encoding="utf-8") as f:
    json.dump(facet_policy_manifest, f, ensure_ascii=False, indent=2)

print("Rows:", len(item_schema_norm))
print("Validation: harmonized Global Review evidence contract built")


In [ ]:
# =========================================================
# Global Review Evidence View Validation
# =========================================================
required_evidence_columns = [
    "brand_facet_text",
    "query_safe_facet_text",
    "specific_query_safe_facet_text",
    "functional_facet_text",
    "canonical_retrieval_text_core",
    "canonical_text_dense_core",
    "canonical_text_sparse_core",
    *HARMONIZED_REVIEW_REPUTATION_COLS,
    "review_reputation_facet_text",
    "review_reputation_only_text",
    "review_reputation_available_flag",
    "review_reputation_signal_family_count",
    "review_reputation_signal_total_count",
    "canonical_retrieval_text",
    "canonical_text_dense",
    "canonical_text_sparse",
    "profile_safe_facet_text",
    "profile_source_text_core",
    "profile_source_text_dedup_seed",
]

missing_evidence_columns = [
    column for column in required_evidence_columns
    if column not in item_schema_norm.columns
]
if missing_evidence_columns:
    raise RuntimeError(f"Missing retrieval evidence columns: {missing_evidence_columns}")

if item_schema_norm["parent_asin"].isna().any() or item_schema_norm["parent_asin"].astype(str).str.strip().eq("").any():
    raise RuntimeError("parent_asin contains null or empty values.")
if not item_schema_norm["parent_asin"].astype(str).is_unique:
    raise RuntimeError("parent_asin must be unique.")

for col in [
    "canonical_retrieval_text_core",
    "canonical_text_dense_core",
    "canonical_text_sparse_core",
    "canonical_retrieval_text",
    "canonical_text_dense",
    "canonical_text_sparse",
]:
    empty_rows = int(item_schema_norm[col].fillna("").astype(str).str.strip().eq("").sum())
    if empty_rows:
        raise RuntimeError(f"{col} must be non-empty for every item. Empty rows: {empty_rows}")

final_differs_from_metadata_rows = int(
    (
        item_schema_norm["canonical_retrieval_text"].fillna("").astype(str)
        != item_schema_norm["canonical_retrieval_text_core"].fillna("").astype(str)
    ).sum()
)
if final_differs_from_metadata_rows <= 0:
    raise RuntimeError("At least one final production representation must differ from metadata-only text.")

for output_col, source_cols in SOURCE_LISTS_USED_FOR_PRODUCTION_CORE.items():
    forbidden_sources = [
        col for col in source_cols
        if col in HISTORICAL_REVIEW_TEXT_COLUMNS
        or col.startswith("historical_review_")
        or col.startswith("review_reputation_")
    ]
    if forbidden_sources:
        raise RuntimeError(f"Historical review sources feed {output_col}: {forbidden_sources}")

brand_sources = set(HARMONIZED_BRAND_SOURCE_COLS + ["facet_brand_text", "brand_facet_text"])
for output_col in ["query_safe_facet_text", "functional_facet_text"]:
    source_cols = set(SOURCE_LISTS_USED_FOR_PRODUCTION_CORE[output_col])
    if brand_sources.intersection(source_cols):
        raise RuntimeError(f"Brand source columns feed {output_col}.")
if "brand_facet_text" not in SOURCE_LISTS_USED_FOR_PRODUCTION_CORE["canonical_retrieval_text_core"]:
    raise RuntimeError("Brand must be included in production retrieval text.")
if "brand_facet_text" not in PROFILE_SOURCE_TEXT_CORE_SOURCE_COLUMNS:
    raise RuntimeError("Brand must be included in the profile source.")

if not PRODUCTION_HISTORICAL_REVIEW_REPUTATION_ENABLED:
    raise RuntimeError("Historical review-reputation must be enabled for production Global Review.")
if not item_schema_norm["review_reputation_only_text"].fillna("").astype(str).str.strip().ne("").any():
    raise RuntimeError("Production Global Review requires non-empty historical review signals.")

print("Rows:", len(item_schema_norm))
print("Validation: Global Review evidence views passed")


## Save harmonized item schema


In [ ]:
# =========================================================
# Output Export
# =========================================================
rating_like_columns = [
    "average_rating",
    "rating_number",
    "rating",
    "rating_count",
    "rating_num",
    "price",
]
item_schema_norm = item_schema_norm.drop(
    columns=[column for column in rating_like_columns if column in item_schema_norm.columns],
    errors="ignore",
)

required_output_columns = [
    "parent_asin",
    "brand_facet_text",
    "facet_brand_text",
    "query_safe_facet_text",
    "specific_query_safe_facet_text",
    "functional_facet_text",
    *HARMONIZED_REVIEW_REPUTATION_COLS,
    "review_reputation_facet_text",
    "review_reputation_only_text",
    "review_reputation_available_flag",
    "review_reputation_signal_family_count",
    "review_reputation_signal_total_count",
    "canonical_metadata_text",
    "canonical_retrieval_text_core",
    "canonical_text_dense_core",
    "canonical_text_sparse_core",
    "canonical_retrieval_text",
    "canonical_text_dense",
    "canonical_text_sparse",
    "profile_safe_facet_text",
    "profile_source_text_core",
    "profile_source_text_dedup_seed",
    "review_reputation_source",
    "review_reputation_cutoff_exclusive",
    "facet_policy_version",
    "evidence_scope",
    "historical_review_reputation_enabled",
    "historical_review_reputation_quarantined",
    "brand_policy",
    "brand_retrieval_enabled",
    "brand_profile_enabled",
    "brand_query_enabled",
    "identifier_policy",
]

missing_output_columns = [
    column for column in required_output_columns
    if column not in item_schema_norm.columns
]
if missing_output_columns:
    raise RuntimeError(f"Missing required output columns: {missing_output_columns}")

item_schema_norm["parent_asin"] = item_schema_norm["parent_asin"].astype(str).str.strip()
if item_schema_norm["parent_asin"].eq("").any() or item_schema_norm["parent_asin"].isna().any():
    raise RuntimeError("parent_asin contains null or empty values.")
if not item_schema_norm["parent_asin"].is_unique:
    raise RuntimeError("parent_asin must be unique.")

for col in ["canonical_retrieval_text", "canonical_text_dense", "canonical_text_sparse"]:
    empty_rows = int(item_schema_norm[col].fillna("").astype(str).str.strip().eq("").sum())
    if empty_rows:
        raise RuntimeError(f"{col} must be non-empty before export. Empty rows: {empty_rows}")

final_differs_from_metadata_rows = int(
    (
        item_schema_norm["canonical_retrieval_text"].fillna("").astype(str)
        != item_schema_norm["canonical_retrieval_text_core"].fillna("").astype(str)
    ).sum()
)
if final_differs_from_metadata_rows <= 0:
    raise RuntimeError("Final production text must differ from metadata-only text for at least one item.")

for raw_col in ["review_text", "review_body", "raw_review_text", "target_review_text", "heldout_review_text"]:
    if raw_col in item_schema_norm.columns:
        raise RuntimeError(f"Raw review text column must not be exported: {raw_col}")

for blocked_col in ["rating", "sentiment"]:
    if blocked_col in item_schema_norm.columns:
        raise RuntimeError(f"{blocked_col} must not be exported or used.")

for count_col in ["historical_review_count_pre_window"]:
    if count_col in item_schema_norm.columns and int(item_schema_norm[count_col].fillna(0).sum()) <= 0:
        raise RuntimeError("Historical review rows before the cutoff must be nonzero.")

for source_col in ["historical_review_reputation_text", "review_reputation_facet_text", "review_reputation_only_text"]:
    if source_col in item_schema_norm.columns and not item_schema_norm[source_col].fillna("").astype(str).str.strip().ne("").any():
        raise RuntimeError(f"{source_col} must be non-empty for at least one item.")

for flag_col in ["review_reputation_available_flag"]:
    if flag_col in item_schema_norm.columns and int(item_schema_norm[flag_col].fillna(False).astype(bool).sum()) <= 0:
        raise RuntimeError("items_with_review_reputation must be greater than zero.")

for count_col in ["review_reputation_signal_family_count", "review_reputation_signal_total_count"]:
    if count_col in item_schema_norm.columns and int(pd.to_numeric(item_schema_norm[count_col], errors="coerce").fillna(0).sum()) <= 0:
        raise RuntimeError(f"{count_col} must be greater than zero across retained items.")

forbidden_exact_columns = {
    "target_review_text",
    "heldout_review_text",
    "review_text",
    "review_body",
    "raw_review_text",
    "rating",
    "sentiment",
    "helpful_vote",
    "prompt",
    "response",
}
forbidden_columns = [
    column for column in item_schema_norm.columns
    if str(column).lower() in forbidden_exact_columns
    or str(column).lower().startswith(("raw_review_", "target_review_", "heldout_review_"))
]
if forbidden_columns:
    raise RuntimeError(f"Forbidden raw or target-review columns in final schema: {forbidden_columns}")

item_schema_norm.to_parquet(SCHEMA_OUTPUT_PATH, index=False)
item_schema_norm.to_parquet(SCHEMA_NORM_OUTPUT_PATH, index=False)
item_schema_norm.to_parquet(SCHEMA_FULL_OUTPUT_PATH, index=False)

if "item_schema_coverage_df" in globals():
    item_schema_coverage_df.to_csv(
        OUTPUT_DIR / "face_item_schema_coverage_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )
if "top_values_norm_df" in globals():
    top_values_norm_df.to_csv(
        OUTPUT_DIR / "face_item_schema_top_values.csv",
        index=False,
        encoding="utf-8-sig",
    )

review_available = item_schema_norm["review_reputation_only_text"].fillna("").str.strip().ne("")
items_with_review_reputation = int(review_available.sum())
review_reputation_item_coverage = float(review_available.mean())
manifest = {
    "items_path": str(ITEMS_PATH),
    "reviews_path": str(REVIEWS_PATH),
    "schema_path": str(SCHEMA_OUTPUT_PATH),
    "rows": int(len(item_schema_norm)),
    "train_review_cutoff_exclusive": TRAIN_REVIEW_CUTOFF_EXCLUSIVE.isoformat(),
    "evidence_scope": PRODUCTION_EVIDENCE_SCOPE,
    "historical_review_reputation_enabled": PRODUCTION_HISTORICAL_REVIEW_REPUTATION_ENABLED,
    "historical_review_reputation_quarantined": HISTORICAL_REVIEW_REPUTATION_QUARANTINED,
    "review_reputation_source": REVIEW_REPUTATION_SOURCE,
    "items_with_review_reputation": items_with_review_reputation,
    "review_reputation_item_coverage": review_reputation_item_coverage,
    "production_text_columns": [
        "canonical_retrieval_text",
        "canonical_text_dense",
        "canonical_text_sparse",
    ],
    "evidence_views": {
        "metadata_functional": "catalog metadata and product-functional facets only",
        "review_only": "historical review-reputation evidence only",
        "global_review": "catalog metadata, product-functional facets, and historical review-reputation evidence",
    },
    "rating_used": False,
    "sentiment_used": False,
    "raw_review_text_exported": False,
    "brand_in_retrieval_text": True,
    "brand_in_profile_source_text": True,
    "brand_in_synthetic_query": False,
}
with open(
    OUTPUT_DIR / "face_feature_engineering_manifest.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)

print("Items with review reputation:", items_with_review_reputation)
print("Review reputation coverage:", review_reputation_item_coverage)
print("Validation: passed")


In [ ]:
# =========================================================
# Subcategory Check
# =========================================================
qc_cols = [
    "title",
    "categories",
    "categories_leaf",
    "sub_category",
    "sub_category_raw",
    "sub_category_norm",
    "sub_category_norm_text",
    "parent_asin",
]

available_qc_cols = [c for c in qc_cols if c in item_schema_norm.columns]
broad_label_tokens = {normalize_sub_category_token(label) for label in GENERIC_SUB_CATEGORY_LABELS}
path_leak_pattern = re.compile(r"[|>]")

def sub_category_raw_has_leak(x):
    values = x if isinstance(x, list) else sanitize_sub_category_values(x)
    for value in values:
        text = clean_text(value)
        if text is None:
            continue
        token = normalize_sub_category_token(text)
        if token in broad_label_tokens:
            return True
        if path_leak_pattern.search(text) is not None:
            return True
    return False

def sub_category_norm_text_has_leak(x):
    text = clean_text(x)
    if text is None:
        return False
    if path_leak_pattern.search(text) is not None:
        return True
    lowered = text.lower()
    return any(label.lower() in lowered for label in GENERIC_SUB_CATEGORY_LABELS)

pass
if "sub_category_raw" in item_schema_norm.columns:
    pass

pass
if "sub_category_norm_text" in item_schema_norm.columns:
    pass

sub_category_raw_leak_count = int(
    item_schema_norm["sub_category_raw"].apply(sub_category_raw_has_leak).sum()
) if "sub_category_raw" in item_schema_norm.columns else 0
sub_category_norm_text_leak_count = int(
    item_schema_norm["sub_category_norm_text"].apply(sub_category_norm_text_has_leak).sum()
) if "sub_category_norm_text" in item_schema_norm.columns else 0

pass
pass

bad_mask = pd.Series(False, index=item_schema_norm.index)
if "sub_category_raw" in item_schema_norm.columns:
    bad_mask |= item_schema_norm["sub_category_raw"].apply(sub_category_raw_has_leak)
if "sub_category_norm_text" in item_schema_norm.columns:
    bad_mask |= item_schema_norm["sub_category_norm_text"].apply(sub_category_norm_text_has_leak)

if bad_mask.any():
    pass
    pass
else:
    pass

pass
pass

In [ ]:
# =========================================================
# Schema Preview
# =========================================================
brand_sample_col = "brand_meta"
sample_cols = ["title", brand_sample_col, "categories", "sub_category_norm_text", "concern_claim_norm_text", "is_discontinued_by_manufacturer_raw", "is_discontinued", "parent_asin"]
available_sample_cols = [c for c in sample_cols if c in item_schema_norm.columns]
sample_n = min(15, len(item_schema_norm))
concern_cols = [c for c in item_schema_norm.columns if c.startswith("concern_claim")]
pass
pass
pass
pass
pass
pass
pass
pass
pass
pass
discontinued_sample_df = item_schema_norm.loc[item_schema_norm["is_discontinued"], available_sample_cols].head(20)
pass
pass

In [ ]:
# =========================================================
# Coverage Summary
# =========================================================
def safe_nunique_for_summary(series):
    try:
        return series.nunique(dropna=True)
    except TypeError:
        normalized = series.map(
            lambda x: json.dumps(x, ensure_ascii=False, sort_keys=True)
            if isinstance(x, (list, dict))
            else x
        )
        return normalized.nunique(dropna=True)

column_summary = pd.DataFrame({
    "column": item_schema_norm.columns,
    "dtype": item_schema_norm.dtypes.astype(str).values,
    "non_null_count": item_schema_norm.notna().sum().values,
    "null_count": item_schema_norm.isna().sum().values,
    "null_ratio": item_schema_norm.isna().mean().values,
    "nunique": [safe_nunique_for_summary(item_schema_norm[c]) for c in item_schema_norm.columns],
}).sort_values(["null_ratio", "nunique"], ascending=[False, False])

major_feature_cols = [
    "brand_meta",
    "sub_category_norm_text",
    "form_norm_text",
    "skin_type_norm_text",
    "ingredient_norm_text",
    "benefit_norm_text",
    "scent_norm_text",
    "free_claim_norm_text",
    "marketing_claim_norm_text",
    "concern_claim_norm_text",
    "product_type_norm_text",
    "texture_norm_text",
    "historical_review_reputation_text",
    "review_reputation_concern_text",
    "review_reputation_skin_type_text",
    "review_reputation_benefit_text",
    "review_reputation_ingredient_text",
    "review_reputation_product_type_or_form_texture_text",
    "brand_facet_text",
    "facet_brand_text",
    "query_safe_facet_text",
    "specific_query_safe_facet_text",
    "functional_facet_text",
    "review_reputation_facet_text",
    "identifier_diagnostic_text",
    "canonical_metadata_text",
    "canonical_retrieval_text_core",
    "canonical_text_dense_core",
    "canonical_text_sparse_core",
    "canonical_retrieval_text",
    "canonical_text_dense",
    "canonical_text_sparse",
    "profile_source_text_core",
    "profile_source_text_dedup_seed",
]
major_feature_cols = [c for c in major_feature_cols if c in item_schema_norm.columns]

coverage_summary = pd.DataFrame({
    "column": major_feature_cols,
    "non_null_count": [item_schema_norm[c].notna().sum() for c in major_feature_cols],
    "coverage_ratio": [item_schema_norm[c].notna().mean() for c in major_feature_cols],
    "nunique": [safe_nunique_for_summary(item_schema_norm[c]) for c in major_feature_cols],
}).sort_values("coverage_ratio", ascending=False)

qc_cols = [
    "parent_asin",
    "title",
    "brand_meta",
    "sub_category_norm_text",
    "form_norm_text",
    "skin_type_norm_text",
    "ingredient_norm_text",
    "benefit_norm_text",
    "scent_norm_text",
    "free_claim_norm_text",
    "marketing_claim_norm_text",
    "concern_claim_norm_text",
    "is_discontinued",
    "historical_review_count_pre_window",
    "historical_review_reputation_text",
    "review_reputation_available_flag",
    "review_reputation_signal_family_count",
    "review_reputation_signal_total_count",
    "facet_brand_text",
    "query_safe_facet_text",
    "canonical_retrieval_text",
    "profile_source_text_dedup_seed",
]
qc_cols = [c for c in qc_cols if c in item_schema_norm.columns]

column_summary.to_csv(OUTPUT_DIR / "face_item_schema_column_summary.csv", index=False, encoding="utf-8-sig")
coverage_summary.to_csv(OUTPUT_DIR / "face_item_schema_coverage_summary.csv", index=False, encoding="utf-8-sig")
item_schema_norm[qc_cols].head(500).to_csv(OUTPUT_DIR / "face_item_schema_qc_preview.csv", index=False, encoding="utf-8-sig")

created_outputs = [
    SCHEMA_OUTPUT_PATH,
    SCHEMA_NORM_OUTPUT_PATH,
    SCHEMA_FULL_OUTPUT_PATH,
    OUTPUT_DIR / "face_item_schema_column_summary.csv",
    OUTPUT_DIR / "face_item_schema_coverage_summary.csv",
    OUTPUT_DIR / "face_item_schema_qc_preview.csv",
    OUTPUT_DIR / "face_item_schema_top_values.csv",
    OUTPUT_DIR / "normalization_maps.json",
    OUTPUT_DIR / "face_item_review_reputation_diagnostics.csv",
    OUTPUT_DIR / "face_item_schema_facet_policy_summary.csv",
    OUTPUT_DIR / "face_feature_engineering_facet_policy_manifest.json",
]

results_overall = pd.DataFrame([
    {"metric": "items_input_rows", "value": len(items)},
    {"metric": "schema_rows", "value": len(item_schema_norm)},
    {"metric": "schema_columns", "value": item_schema_norm.shape[1]},
    {"metric": "unique_parent_asin", "value": item_schema_norm["parent_asin"].astype(str).nunique()},
    {"metric": "duplicate_parent_asin_rows", "value": int(item_schema_norm["parent_asin"].astype(str).duplicated().sum())},
    {"metric": "discontinued_rows", "value": int(item_schema_norm["is_discontinued"].sum())},
    {"metric": "items_with_historical_reputation", "value": int(item_schema_norm["review_reputation_available_flag"].sum())},
])

diagnostics_summary = pd.DataFrame([
    {"check": "items_path_exists", "value": ITEMS_PATH.exists()},
    {"check": "raw_reviews_loaded_only_for_historical_item_reputation", "value": REVIEW_REPUTATION_ENABLED and "raw_reviews_for_reputation" not in globals()},
    {"check": "schema_has_rows", "value": len(item_schema_norm) > 0},
    {"check": "parent_asin_non_null", "value": not item_schema_norm["parent_asin"].isna().any()},
    {"check": "parent_asin_unique", "value": item_schema_norm["parent_asin"].astype(str).is_unique},
    {"check": "column_summary_rows", "value": len(column_summary)},
    {"check": "coverage_summary_rows", "value": len(coverage_summary)},
    {"check": "parent_asin_required_no_asin_fallback_for_reputation", "value": PARENT_ASIN_REQUIRED_NO_ASIN_FALLBACK_FOR_REPUTATION},
    {"check": "historical_reviews_restricted_to_cutoff", "value": HISTORICAL_REVIEWS_RESTRICTED_TO_CUTOFF},
    {"check": "safe_review_reputation_prefix_allowed", "value": SAFE_REVIEW_REPUTATION_PREFIX_ALLOWED},
    {"check": "unsafe_target_review_columns_still_forbidden", "value": UNSAFE_TARGET_REVIEW_COLUMNS_STILL_FORBIDDEN},
    {"check": "production_evidence_scope_global_review", "value": PRODUCTION_EVIDENCE_SCOPE == "catalog_metadata_functional_facets_and_historical_review_signals"},
    {"check": "historical_review_reputation_enabled_for_production", "value": PRODUCTION_HISTORICAL_REVIEW_REPUTATION_ENABLED},
    {"check": "production_text_differs_from_metadata_only", "value": int((item_schema_norm["canonical_retrieval_text"].fillna("").astype(str) != item_schema_norm["canonical_retrieval_text_core"].fillna("").astype(str)).sum()) > 0},
])

pass
pass
pass
pass
pass
pass
pass

pass
pass
pass
pass
pass

# Common cross-category schema framework


In [ ]:
# =========================================================
# Common Cross-Category Schema Framework
# =========================================================
COMMON_SCHEMA_POLICY_VERSION = "common_schema_roles_v6_global_review_brand_retrieval_profile"
COMMON_EVIDENCE_SCOPE = PRODUCTION_EVIDENCE_SCOPE
COMMON_HISTORICAL_REVIEW_REPUTATION_ENABLED = PRODUCTION_HISTORICAL_REVIEW_REPUTATION_ENABLED

COMMON_SCHEMA_ROLES = [
    "brand",
    "category_or_product_type",
    "form_texture",
    "ingredient_or_composition",
    "need_benefit_concern",
    "claim_constraint",
    "target_context",
    "sensory",
    "review_derived_signal",
]
COMMON_QUERY_SAFE_ROLES = [
    "category_or_product_type",
    "form_texture",
    "ingredient_or_composition",
    "need_benefit_concern",
    "claim_constraint",
    "target_context",
    "sensory",
]
COMMON_PROFILE_ROLES = ["brand"] + COMMON_QUERY_SAFE_ROLES

ROLE_SOURCE_MAP = {
    "brand": ["brand_facet_text", "facet_brand_text", "brand_meta", "brand"],
    "category_or_product_type": [
        "facet_category_text",
        "sub_category_norm_text",
        "sub_category_text",
        "categories_leaf_text",
    ],
    "form_texture": [
        "facet_form_text",
        "form_norm_text",
        "form_text",
        "texture_norm_text",
    ],
    "ingredient_or_composition": [
        "facet_ingredient_text",
        "ingredient_norm_text",
        "ingredient_text",
    ],
    "need_benefit_concern": [
        "facet_benefit_text",
        "benefit_norm_text",
        "benefit_text",
        "concern_claim_norm_text",
    ],
    "claim_constraint": [
        "facet_claim_text",
        "free_claim_norm_text",
        "marketing_claim_norm_text",
        "free_claim_text",
    ],
    "target_context": [
        "facet_skin_type_text",
        "skin_type_norm_text",
        "skin_type_text",
    ],
    "sensory": [
        "facet_scent_text",
        "scent_norm_text",
        "scent_text",
    ],
    "review_derived_signal": [
        "review_reputation_facet_text",
        "historical_review_reputation_text",
        "review_reputation_concern_text",
        "review_reputation_skin_type_text",
        "review_reputation_benefit_text",
        "review_reputation_ingredient_text",
        "review_reputation_product_type_or_form_texture_text",
    ],
}

GENERIC_CATEGORY_ANCHORS_COMMON = {
    "skin care",
    "skincare",
    "face",
    "facial",
    "skin care routine",
    "skincare routine",
}
GENERIC_UTILITY_TOKENS_COMMON = {
    "support",
    "supports",
    "help",
    "helps",
    "promote",
    "promotes",
    "boost",
    "formula",
    "blend",
    "complex",
    "product",
    "solution",
}
CONTEXT_DEPENDENT_UTILITY_TOKENS_COMMON = {
    "daily",
    "natural",
    "wellness",
    "health",
    "care",
    "routine",
}
SPECIFIC_PHRASE_EXCEPTIONS_COMMON = {
    "morning routine",
    "night routine",
    "skin barrier support",
    "hydration support",
    "sensitive skin care",
    "acne care",
}


def common_text(value):
    if value is None:
        return ""
    if isinstance(value, (list, tuple, set, np.ndarray, pd.Series)):
        return " | ".join(common_text(item) for item in value if common_text(item))
    if isinstance(value, dict):
        return " | ".join(common_text(item) for item in value.values() if common_text(item))
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    text = re.sub(r"\s+", " ", str(value).replace("\n", " ").replace("\t", " ")).strip()
    if text.lower() in {"", "none", "null", "nan", "n/a", "na", "[]", "{}"}:
        return ""
    return text


def common_split(value):
    text = common_text(value)
    if not text:
        return []
    return [
        part
        for part in (
            common_text(piece)
            for piece in re.split(r"\s*\|\s*|\s*;\s*", text)
        )
        if part
    ]


def common_join(values):
    output = []
    seen = set()
    for value in values:
        for part in common_split(value):
            key = part.lower()
            if key not in seen:
                seen.add(key)
                output.append(part)
    return " | ".join(output)


def common_join_cols(row, columns):
    return common_join(row[column] for column in columns if column in row.index)


item_schema_common_df = item_schema_norm.copy()
item_schema_common_df["parent_asin"] = item_schema_common_df["parent_asin"].astype(str).str.strip()
if item_schema_common_df["parent_asin"].eq("").any():
    raise RuntimeError("parent_asin contains empty values.")
if not item_schema_common_df["parent_asin"].is_unique:
    raise RuntimeError("parent_asin must be unique.")

role_map_rows = []
for role in COMMON_SCHEMA_ROLES:
    requested_columns = ROLE_SOURCE_MAP[role]
    available_columns = [
        column for column in requested_columns
        if column in item_schema_common_df.columns
    ]
    output_column = f"common_{role}_text"
    item_schema_common_df[output_column] = item_schema_common_df.apply(
        lambda row, columns=available_columns: common_join_cols(row, columns),
        axis=1,
    )
    role_map_rows.append({
        "common_role": role,
        "output_column": output_column,
        "source_columns": " | ".join(available_columns),
        "missing_source_columns": " | ".join(
            column for column in requested_columns
            if column not in available_columns
        ),
        "query_safe_role": role in COMMON_QUERY_SAFE_ROLES,
        "profile_role": role in COMMON_PROFILE_ROLES,
        "retrieval_role": role in COMMON_QUERY_SAFE_ROLES or role in {"brand", "review_derived_signal"},
        "brand_role": role == "brand",
        "product_functional_role": role in COMMON_QUERY_SAFE_ROLES,
        "review_derived_role": role == "review_derived_signal",
        "policy_version": COMMON_SCHEMA_POLICY_VERSION,
    })

common_schema_role_map = pd.DataFrame(role_map_rows)
query_safe_columns = [f"common_{role}_text" for role in COMMON_QUERY_SAFE_ROLES]
item_schema_common_df["common_query_safe_facet_text"] = item_schema_common_df.apply(
    lambda row: common_join_cols(row, query_safe_columns),
    axis=1,
)

generic_anchor_norms = {value.lower() for value in GENERIC_CATEGORY_ANCHORS_COMMON}
generic_utility_norms = {value.lower() for value in GENERIC_UTILITY_TOKENS_COMMON}
context_utility_norms = {value.lower() for value in CONTEXT_DEPENDENT_UTILITY_TOKENS_COMMON}
specific_exception_norms = {value.lower() for value in SPECIFIC_PHRASE_EXCEPTIONS_COMMON}


def partition_common_facet_text(value):
    generic_anchors = []
    generic_utilities = []
    context_utilities = []
    specific_values = []

    for part in common_split(value):
        normalized = part.lower()
        if normalized in specific_exception_norms:
            specific_values.append(part)
        elif normalized in generic_anchor_norms:
            generic_anchors.append(part)
        elif normalized in generic_utility_norms:
            generic_utilities.append(part)
        elif normalized in context_utility_norms:
            context_utilities.append(part)
        else:
            specific_values.append(part)

    return pd.Series({
        "common_generic_anchor_text": common_join(generic_anchors),
        "common_generic_utility_text": common_join(generic_utilities),
        "common_context_dependent_utility_text": common_join(context_utilities),
        "common_specific_query_safe_facet_text": common_join(specific_values),
    })


partition_cols = [
    "common_generic_anchor_text",
    "common_generic_utility_text",
    "common_context_dependent_utility_text",
    "common_specific_query_safe_facet_text",
]
item_schema_common_df = item_schema_common_df.drop(columns=partition_cols, errors="ignore")
item_schema_common_df = pd.concat(
    [item_schema_common_df, item_schema_common_df["common_query_safe_facet_text"].apply(partition_common_facet_text)],
    axis=1,
)

item_schema_common_df["common_brand_facet_text"] = item_schema_common_df["common_brand_text"]
item_schema_common_df["common_functional_facet_text"] = item_schema_common_df["common_specific_query_safe_facet_text"]
item_schema_common_df["common_profile_safe_facet_text"] = item_schema_common_df.apply(
    lambda row: common_join([
        row.get("common_brand_facet_text"),
        row.get("common_functional_facet_text"),
    ]),
    axis=1,
)
item_schema_common_df["common_profile_schema_text"] = item_schema_common_df["common_profile_safe_facet_text"]
item_schema_common_df["common_canonical_metadata_text"] = item_schema_common_df["canonical_metadata_text"]
item_schema_common_df["common_canonical_retrieval_text_core"] = item_schema_common_df["canonical_retrieval_text_core"]
item_schema_common_df["common_canonical_text_dense_core"] = item_schema_common_df["canonical_text_dense_core"]
item_schema_common_df["common_canonical_text_sparse_core"] = item_schema_common_df["canonical_text_sparse_core"]
item_schema_common_df["common_review_reputation_only_text"] = item_schema_common_df["review_reputation_only_text"]
item_schema_common_df["common_canonical_retrieval_text"] = item_schema_common_df["canonical_retrieval_text"]
item_schema_common_df["common_canonical_text_dense"] = item_schema_common_df["canonical_text_dense"]
item_schema_common_df["common_canonical_text_sparse"] = item_schema_common_df["canonical_text_sparse"]
item_schema_common_df["common_profile_source_text_core"] = item_schema_common_df["profile_source_text_core"]
item_schema_common_df["common_profile_source_text_dedup_seed"] = item_schema_common_df["profile_source_text_dedup_seed"]
item_schema_common_df["common_schema_policy_version"] = COMMON_SCHEMA_POLICY_VERSION
item_schema_common_df["common_evidence_scope"] = COMMON_EVIDENCE_SCOPE
item_schema_common_df["common_historical_review_reputation_enabled"] = COMMON_HISTORICAL_REVIEW_REPUTATION_ENABLED

if not item_schema_common_df["common_canonical_retrieval_text"].fillna("").astype(str).str.strip().ne("").all():
    raise RuntimeError("common_canonical_retrieval_text must be non-empty.")

role_coverage_rows = []
for role in COMMON_SCHEMA_ROLES:
    column = f"common_{role}_text"
    non_empty = item_schema_common_df[column].fillna("").astype(str).str.strip().ne("")
    distinct_values = set()
    for value in item_schema_common_df.loc[non_empty, column]:
        distinct_values.update(part.lower() for part in common_split(value))
    role_coverage_rows.append({
        "common_role": role,
        "item_count": int(non_empty.sum()),
        "coverage_rate": float(non_empty.mean()),
        "distinct_value_count": int(len(distinct_values)),
        "is_product_functional_facet": role in COMMON_QUERY_SAFE_ROLES,
    })

common_schema_role_coverage = pd.DataFrame(role_coverage_rows)
common_schema_role_coverage_no_brand = common_schema_role_coverage.loc[
    common_schema_role_coverage["common_role"].ne("brand")
].copy()
functional_coverage = common_schema_role_coverage.loc[
    common_schema_role_coverage["is_product_functional_facet"]
].copy()
common_schema_uniformity_summary = pd.DataFrame([{
    "core_coverage_mean": float(functional_coverage["coverage_rate"].mean()),
    "core_coverage_min": float(functional_coverage["coverage_rate"].min()),
    "core_coverage_max": float(functional_coverage["coverage_rate"].max()),
    "core_coverage_range": float(functional_coverage["coverage_rate"].max() - functional_coverage["coverage_rate"].min()),
    "core_coverage_cv": float(functional_coverage["coverage_rate"].std(ddof=0) / functional_coverage["coverage_rate"].mean()) if functional_coverage["coverage_rate"].mean() else np.nan,
}])

facet_rows = []
for row in item_schema_common_df.itertuples(index=False):
    row_dict = row._asdict()
    for role in COMMON_SCHEMA_ROLES:
        for value in common_split(row_dict.get(f"common_{role}_text", "")):
            normalized = value.lower()
            is_brand = role == "brand"
            is_review_derived = role == "review_derived_signal"
            is_generic_anchor = normalized in generic_anchor_norms
            is_generic_utility = normalized in generic_utility_norms
            is_context_utility = normalized in context_utility_norms
            is_exception = normalized in specific_exception_norms
            is_review_graph_safe = (
                is_review_derived
                and not is_generic_anchor
                and not is_generic_utility
                and not is_context_utility
            )
            is_brand_graph_safe = (
                is_brand
                and not is_generic_anchor
                and not is_generic_utility
                and not is_context_utility
            )
            is_specific = (
                role in COMMON_QUERY_SAFE_ROLES
                and not is_brand
                and not is_review_derived
                and not is_generic_anchor
                and not is_generic_utility
                and (not is_context_utility or is_exception)
            )
            facet_rows.append({
                "parent_asin": row_dict["parent_asin"],
                "common_role": role,
                "facet_role": role,
                "facet_family": role,
                "facet_value": value,
                "facet_value_norm": normalized,
                "is_brand": is_brand,
                "is_review_derived": is_review_derived,
                "is_generic_category_anchor": is_generic_anchor,
                "is_generic_utility_token": is_generic_utility,
                "is_context_dependent_utility_token": is_context_utility,
                "is_specific_phrase_exception": is_exception,
                "is_specific_facet_phrase": is_specific,
                "is_query_safe": is_specific,
                "is_product_functional_facet": is_specific,
                "is_retrieval_safe": bool(is_brand_graph_safe or is_specific or is_review_graph_safe),
                "is_profile_safe": bool(is_brand_graph_safe or is_specific),
                "policy_version": COMMON_SCHEMA_POLICY_VERSION,
            })

items_facets_common = pd.DataFrame(facet_rows)
if items_facets_common.empty:
    raise RuntimeError("Common item facet table is empty.")

facet_vocab_common = (
    items_facets_common
    .groupby(["common_role", "facet_value_norm"], as_index=False)
    .agg(
        facet_value=("facet_value", "first"),
        item_count=("parent_asin", "nunique"),
        is_query_safe=("is_query_safe", "max"),
        is_brand=("is_brand", "max"),
        is_review_derived=("is_review_derived", "max"),
        is_product_functional_facet=("is_product_functional_facet", "max"),
        is_retrieval_safe=("is_retrieval_safe", "max"),
        is_profile_safe=("is_profile_safe", "max"),
        is_generic_category_anchor=("is_generic_category_anchor", "max"),
        is_generic_utility_token=("is_generic_utility_token", "max"),
        is_context_dependent_utility_token=("is_context_dependent_utility_token", "max"),
        is_specific_facet_phrase=("is_specific_facet_phrase", "max"),
    )
    .sort_values(["common_role", "item_count", "facet_value_norm"], ascending=[True, False, True])
)

brand_policy_summary_df = pd.DataFrame([{
    "brand_policy": "Brand is retained as a separate query-unsafe preference facet and is active in retrieval, graph, and user-profile representations.",
    "brand_coverage_rate": float(common_schema_role_coverage.loc[common_schema_role_coverage["common_role"].eq("brand"), "coverage_rate"].iloc[0]),
    "functional_facet_coverage_rate": float(functional_coverage["coverage_rate"].mean()),
}])

common_schema_diagnostics = pd.DataFrame([
    {"check": "parent_asin_unique", "value": bool(item_schema_common_df["parent_asin"].is_unique)},
    {"check": "evidence_scope_global_review", "value": COMMON_EVIDENCE_SCOPE == "catalog_metadata_functional_facets_and_historical_review_signals"},
    {"check": "historical_review_reputation_enabled_for_production", "value": COMMON_HISTORICAL_REVIEW_REPUTATION_ENABLED},
    {"check": "common_canonical_text_non_empty", "value": bool(item_schema_common_df["common_canonical_retrieval_text"].fillna("").astype(str).str.strip().ne("").all())},
    {"check": "brand_not_product_functional", "value": bool(not items_facets_common.loc[items_facets_common["is_brand"], "is_product_functional_facet"].any())},
    {"check": "brand_not_query_safe", "value": bool(not items_facets_common.loc[items_facets_common["is_brand"], "is_query_safe"].any())},
    {"check": "brand_retrieval_safe", "value": bool(items_facets_common.loc[items_facets_common["is_brand"], "is_retrieval_safe"].all())},
    {"check": "brand_profile_safe", "value": bool(items_facets_common.loc[items_facets_common["is_brand"], "is_profile_safe"].all())},
    {"check": "review_derived_not_product_functional", "value": bool(not items_facets_common.loc[items_facets_common["is_review_derived"], "is_product_functional_facet"].any())},
    {"check": "review_derived_not_query_safe", "value": bool(not items_facets_common.loc[items_facets_common["is_review_derived"], "is_query_safe"].any())},
    {"check": "generic_anchor_not_specific", "value": bool(not items_facets_common.loc[items_facets_common["is_generic_category_anchor"], "is_specific_facet_phrase"].any())},
    {"check": "generic_utility_not_specific", "value": bool(not items_facets_common.loc[items_facets_common["is_generic_utility_token"], "is_specific_facet_phrase"].any())},
    {"check": "context_utility_not_specific_without_exception", "value": bool(not items_facets_common.loc[items_facets_common["is_context_dependent_utility_token"] & ~items_facets_common["is_specific_phrase_exception"], "is_specific_facet_phrase"].any())},
])

item_schema_common_df.to_parquet(COMMON_ITEM_SCHEMA_PATH, index=False)
items_facets_common.to_parquet(ITEM_FACETS_PATH, index=False)
facet_vocab_common.to_parquet(FACET_VOCAB_PATH, index=False)
item_schema_common_df.to_parquet(SCHEMA_OUTPUT_PATH, index=False)
item_schema_common_df.to_parquet(SCHEMA_NORM_OUTPUT_PATH, index=False)
item_schema_common_df.to_parquet(SCHEMA_FULL_OUTPUT_PATH, index=False)
item_schema_norm = item_schema_common_df

common_schema_role_map.to_csv(OUTPUT_DIR / "face_common_schema_role_map.csv", index=False, encoding="utf-8-sig")
common_schema_role_coverage.to_csv(OUTPUT_DIR / "face_common_schema_role_coverage.csv", index=False, encoding="utf-8-sig")
common_schema_role_coverage_no_brand.to_csv(OUTPUT_DIR / "face_common_schema_role_coverage_no_brand.csv", index=False, encoding="utf-8-sig")
common_schema_uniformity_summary.to_csv(OUTPUT_DIR / "face_common_schema_uniformity_summary.csv", index=False, encoding="utf-8-sig")
common_schema_diagnostics.to_csv(OUTPUT_DIR / "face_common_schema_diagnostics.csv", index=False, encoding="utf-8-sig")
brand_policy_summary_df.to_csv(OUTPUT_DIR / "face_brand_policy_summary.csv", index=False, encoding="utf-8-sig")

common_schema_contract = {
    "common_item_schema_path": str(COMMON_ITEM_SCHEMA_PATH),
    "items_facets_path": str(ITEM_FACETS_PATH),
    "facet_vocab_path": str(FACET_VOCAB_PATH),
    "policy_version": COMMON_SCHEMA_POLICY_VERSION,
    "evidence_scope": COMMON_EVIDENCE_SCOPE,
    "historical_review_reputation_enabled": COMMON_HISTORICAL_REVIEW_REPUTATION_ENABLED,
    "historical_review_reputation_quarantined": HISTORICAL_REVIEW_REPUTATION_QUARANTINED,
    "production_text_columns": ["canonical_retrieval_text", "canonical_text_dense", "canonical_text_sparse"],
    "common_roles": COMMON_SCHEMA_ROLES,
    "query_safe_roles": COMMON_QUERY_SAFE_ROLES,
    "profile_roles": COMMON_PROFILE_ROLES,
    "role_source_map": ROLE_SOURCE_MAP,
    "generic_category_anchors": sorted(GENERIC_CATEGORY_ANCHORS_COMMON),
    "generic_utility_tokens": sorted(GENERIC_UTILITY_TOKENS_COMMON),
    "context_dependent_utility_tokens": sorted(CONTEXT_DEPENDENT_UTILITY_TOKENS_COMMON),
    "specific_phrase_exceptions": sorted(SPECIFIC_PHRASE_EXCEPTIONS_COMMON),
    "brand_policy": "query_unsafe_retrieval_profile_graph_safe",
    "review_signal_policy": "historical_review_reputation_used_as_global_item_side_retrieval_evidence_only",
}
with open(OUTPUT_DIR / "face_common_schema_contract.json", "w", encoding="utf-8") as file:
    json.dump(common_schema_contract, file, ensure_ascii=False, indent=2)

print("Output:", COMMON_ITEM_SCHEMA_PATH)
print("Rows:", len(item_schema_common_df))
print("Validation: common Global Review schema export passed")


In [ ]:
# =========================================================
# Common Schema Reload Check
# =========================================================
common_required_cols = [
    "common_brand_text",
    "common_category_or_product_type_text",
    "common_form_texture_text",
    "common_ingredient_or_composition_text",
    "common_need_benefit_concern_text",
    "common_claim_constraint_text",
    "common_target_context_text",
    "common_sensory_text",
    "common_review_derived_signal_text",
    "common_query_safe_facet_text",
    "common_generic_anchor_text",
    "common_generic_utility_text",
    "common_context_dependent_utility_text",
    "common_specific_query_safe_facet_text",
    "common_brand_facet_text",
    "common_functional_facet_text",
    "common_profile_safe_facet_text",
    "common_profile_schema_text",
    "common_canonical_metadata_text",
    "common_canonical_retrieval_text_core",
    "common_canonical_text_dense_core",
    "common_canonical_text_sparse_core",
    "common_review_reputation_only_text",
    "common_canonical_retrieval_text",
    "common_canonical_text_dense",
    "common_canonical_text_sparse",
    "common_profile_source_text_core",
    "common_profile_source_text_dedup_seed",
    "common_schema_policy_version",
    "common_evidence_scope",
    "common_historical_review_reputation_enabled",
]

for path in [COMMON_ITEM_SCHEMA_PATH, globals().get("SCHEMA_OUTPUT_PATH", COMMON_ITEM_SCHEMA_PATH)]:
    if not path:
        continue
    df_reload = pd.read_parquet(path)
    missing_common_cols = sorted(set(common_required_cols) - set(df_reload.columns))
    print("Output:", path)
    print("Rows:", len(df_reload))
    print("Validation: common schema columns checked")
    if missing_common_cols:
        raise RuntimeError(f"Missing common schema columns in {path}: {missing_common_cols}")
    if df_reload["common_canonical_retrieval_text"].fillna("").astype(str).str.strip().eq("").any():
        raise RuntimeError(f"common_canonical_retrieval_text must be non-empty in {path}.")

facets_reload = pd.read_parquet(ITEM_FACETS_PATH)
vocab_reload = pd.read_parquet(FACET_VOCAB_PATH)
print("Output:", ITEM_FACETS_PATH)
print("Rows:", len(facets_reload))
print("Output:", FACET_VOCAB_PATH)
print("Rows:", len(vocab_reload))

if facets_reload.empty:
    raise RuntimeError("Common item facets reload is empty.")
if vocab_reload.empty:
    raise RuntimeError("Common facet vocab reload is empty.")


## Downstream schema reload QC


In [ ]:
# =========================================================
# Reload Check
# =========================================================
required_cols = [
    "brand_facet_text",
    "facet_brand_text",
    "query_safe_facet_text",
    "specific_query_safe_facet_text",
    "functional_facet_text",
    "review_reputation_facet_text",
    "identifier_diagnostic_text",
    "canonical_metadata_text",
    "canonical_retrieval_text_core",
    "canonical_text_dense_core",
    "canonical_text_sparse_core",
    "review_reputation_only_text",
    "canonical_retrieval_text",
    "canonical_text_dense",
    "canonical_text_sparse",
    "profile_safe_facet_text",
    "profile_source_text_core",
    "profile_source_text_dedup_seed",
    "evidence_scope",
    "historical_review_reputation_enabled",
    "facet_policy_version",
    "brand_policy",
    "brand_retrieval_enabled",
    "brand_profile_enabled",
    "brand_query_enabled",
    "identifier_policy",
]

for path in [SCHEMA_OUTPUT_PATH, SCHEMA_NORM_OUTPUT_PATH, SCHEMA_FULL_OUTPUT_PATH]:
    df = pd.read_parquet(path)
    missing = sorted(set(required_cols) - set(df.columns))
    print("Output:", path)
    print("Rows:", len(df))
    print("Validation: downstream schema columns checked")
    if missing:
        raise RuntimeError(f"Missing required downstream columns in {path}: {missing}")

    if df["canonical_retrieval_text"].fillna("").astype(str).str.strip().eq("").any():
        raise RuntimeError(f"canonical_retrieval_text must be non-empty in {path}.")

    text_cols = [
        "brand_facet_text",
        "query_safe_facet_text",
        "specific_query_safe_facet_text",
        "functional_facet_text",
        "review_reputation_facet_text",
        "identifier_diagnostic_text",
        "canonical_metadata_text",
        "canonical_retrieval_text_core",
        "canonical_text_dense_core",
        "canonical_text_sparse_core",
        "canonical_retrieval_text",
        "profile_safe_facet_text",
        "profile_source_text_dedup_seed",
    ]
    missingness = df[text_cols].replace("", pd.NA).isna().mean().sort_values(ascending=False)

print("Validation: Global Review schema reload passed")


In [ ]:
# =========================================================
# QC
# =========================================================
def dbg_clean_text(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    text = re.sub(r"\s+", " ", str(value).replace("\n", " ").replace("\t", " ")).strip()
    if text.lower() in {"", "none", "null", "nan", "n/a", "na", "[]", "{}"}:
        return None
    return text


def dbg_split_values(value):
    values = []
    for part in re.split(r"\s*\|\s*|\s*;\s*|\s*,\s*", str(value or "")):
        cleaned = dbg_clean_text(part)
        if cleaned is not None:
            values.append(cleaned)
    return values


def dbg_norm_set(value):
    return {v.lower() for v in dbg_split_values(value)}


schema_debug_df = item_schema_norm.copy()

required_for_notebook_04 = [
    "parent_asin",
    "title",
    "facet_brand_text",
    "brand_facet_text",
    "query_safe_facet_text",
    "specific_query_safe_facet_text",
    "functional_facet_text",
    "review_reputation_facet_text",
    "review_reputation_only_text",
    "historical_review_reputation_text",
    "canonical_retrieval_text",
    "canonical_text_dense",
    "canonical_text_sparse",
    "profile_safe_facet_text",
    "profile_source_text_dedup_seed",
    "evidence_scope",
    "historical_review_reputation_enabled",
    "facet_policy_version",
    "brand_policy",
    "brand_retrieval_enabled",
    "brand_profile_enabled",
    "brand_query_enabled",
    "identifier_policy",
]

missing_for_04 = [
    col for col in required_for_notebook_04
    if col not in schema_debug_df.columns
]

query_safe_cols_debug = [
    col for col in [
        "facet_category_text",
        "facet_product_type_text",
        "facet_form_text",
        "facet_formulation_text",
        "facet_texture_text",
        "facet_skin_type_text",
        "facet_ingredient_text",
        "facet_concern_text",
        "facet_benefit_text",
        "facet_usage_target_text",
        "facet_scent_text",
        "facet_claim_text",
    ]
    if col in schema_debug_df.columns
]

brand_in_query_safe_source_rows = int(
    schema_debug_df.apply(
        lambda row: any(
            bool(dbg_norm_set(row.get(col)) & dbg_norm_set(row.get("brand_facet_text")))
            for col in query_safe_cols_debug
        ),
        axis=1,
    ).sum()
)

brand_in_query_safe_text_rows = int(
    schema_debug_df.apply(
        lambda row: bool(
            dbg_norm_set(row.get("query_safe_facet_text"))
            & dbg_norm_set(row.get("brand_facet_text"))
        ),
        axis=1,
    ).sum()
)

brand_in_specific_rows = int(
    schema_debug_df.apply(
        lambda row: bool(
            dbg_norm_set(row.get("specific_query_safe_facet_text"))
            & dbg_norm_set(row.get("brand_facet_text"))
        ),
        axis=1,
    ).sum()
)

brand_in_functional_rows = int(
    schema_debug_df.apply(
        lambda row: bool(
            dbg_norm_set(row.get("functional_facet_text"))
            & dbg_norm_set(row.get("brand_facet_text"))
        ),
        axis=1,
    ).sum()
)

brand_rows_debug = schema_debug_df["brand_facet_text"].fillna("").astype(str).str.strip().ne("")
brand_missing_retrieval_rows = int(
    schema_debug_df.loc[brand_rows_debug].apply(
        lambda row: not dbg_norm_set(row.get("brand_facet_text")).issubset(
            dbg_norm_set(row.get("canonical_retrieval_text"))
        ),
        axis=1,
    ).sum()
)
brand_missing_profile_rows = int(
    schema_debug_df.loc[brand_rows_debug].apply(
        lambda row: not dbg_norm_set(row.get("brand_facet_text")).issubset(
            dbg_norm_set(row.get("profile_safe_facet_text"))
        ),
        axis=1,
    ).sum()
)

retrieval_empty_rows = int(
    schema_debug_df["canonical_retrieval_text"].fillna("").astype(str).str.strip().eq("").sum()
)
dense_empty_rows = int(
    schema_debug_df["canonical_text_dense"].fillna("").astype(str).str.strip().eq("").sum()
)
sparse_empty_rows = int(
    schema_debug_df["canonical_text_sparse"].fillna("").astype(str).str.strip().eq("").sum()
)

production_differs_from_metadata_rows = int(
    (
        schema_debug_df["canonical_retrieval_text"].fillna("").astype(str)
        != schema_debug_df["canonical_retrieval_text_core"].fillna("").astype(str)
    ).sum()
) if "canonical_retrieval_text_core" in schema_debug_df.columns else -1

debug_handoff_checks = pd.DataFrame([
    {
        "check": "Notebook 04 required columns present",
        "value": len(missing_for_04),
        "expected": 0,
        "pass": len(missing_for_04) == 0,
    },
    {
        "check": "query-safe source brand collisions",
        "value": brand_in_query_safe_source_rows,
        "expected": 0,
        "pass": brand_in_query_safe_source_rows == 0,
    },
    {
        "check": "query_safe_facet_text brand collisions",
        "value": brand_in_query_safe_text_rows,
        "expected": 0,
        "pass": brand_in_query_safe_text_rows == 0,
    },
    {
        "check": "specific_query_safe_facet_text brand collisions",
        "value": brand_in_specific_rows,
        "expected": 0,
        "pass": brand_in_specific_rows == 0,
    },
    {
        "check": "functional_facet_text brand collisions",
        "value": brand_in_functional_rows,
        "expected": 0,
        "pass": brand_in_functional_rows == 0,
    },
    {
        "check": "brand included in retrieval text",
        "value": brand_missing_retrieval_rows,
        "expected": 0,
        "pass": brand_missing_retrieval_rows == 0,
    },
    {
        "check": "brand included in profile-safe facets",
        "value": brand_missing_profile_rows,
        "expected": 0,
        "pass": brand_missing_profile_rows == 0,
    },
    {
        "check": "canonical_retrieval_text non-empty",
        "value": retrieval_empty_rows,
        "expected": 0,
        "pass": retrieval_empty_rows == 0,
    },
    {
        "check": "canonical_text_dense non-empty",
        "value": dense_empty_rows,
        "expected": 0,
        "pass": dense_empty_rows == 0,
    },
    {
        "check": "canonical_text_sparse non-empty",
        "value": sparse_empty_rows,
        "expected": 0,
        "pass": sparse_empty_rows == 0,
    },
    {
        "check": "final production text differs from metadata-only text",
        "value": production_differs_from_metadata_rows,
        "expected": "> 0",
        "pass": production_differs_from_metadata_rows > 0,
    },
])

display(debug_handoff_checks)

if missing_for_04:
    raise RuntimeError(f"Missing Notebook 04 required columns: {missing_for_04}")

failed = debug_handoff_checks.loc[~debug_handoff_checks["pass"]]
if len(failed):
    display(failed)
    raise RuntimeError("Notebook 02 -> Notebook 04 handoff checks failed.")

print("Notebook 02 handoff checks passed. You can continue with Notebook 04.")
print("Schema output:", SCHEMA_OUTPUT_PATH)
print("Rows:", len(schema_debug_df))
